<div style="background:#0d1117; border-radius:12px; padding:40px 44px 32px; border:1.5px solid #30363d; font-family:'Segoe UI',sans-serif;">

<div style="display:flex; align-items:center; gap:14px; margin-bottom:24px;">
  <div style="width:6px; height:72px; background:linear-gradient(180deg,#39d353,#00bcd4); border-radius:3px;"></div>
  <div>
    <h1 style="color:#ffffff; font-size:3.1em; margin:0; letter-spacing:0.5px;">MSME Credit Invisibility Score</h1>
    <p style="color:#39d353; font-size:1.05em; margin:6px 0 0; font-style:italic;">India's First State-Level Alternative Credit Signal Framework</p>
  </div>
</div>

<table style="border-collapse:collapse; width:100%; background:#161b22; border-radius:8px; overflow:hidden;">
<tr>
  <td style="padding:10px 18px; color:#8b949e; font-size:0.88em; width:18%;"> Author</td>
  <td style="padding:10px 18px; color:#e6edf3; font-weight:600;">Om Dadhe — Data &amp; Business Analyst</td>
  <td style="padding:10px 18px; color:#8b949e; font-size:0.88em; width:15%;"> Affiliation</td>
  <td style="padding:10px 18px; color:#e6edf3;">GITAM University Hyderabad, BTech CSE (May 2026)</td>
</tr>
<tr style="background:#21262d;">
  <td style="padding:10px 18px; color:#8b949e; font-size:0.88em;"> Date</td>
  <td style="padding:10px 18px; color:#e6edf3;">December 2025</td>
  <td style="padding:10px 18px; color:#8b949e; font-size:0.88em;"> Portfolio</td>
  <td style="padding:10px 18px; color:#e6edf3;"><a href="https://om-dadhe-portfolio.vercel.app" style="color:#00bcd4;">om-dadhe-portfolio.vercel.app</a> &nbsp;·&nbsp; <a href="https://github.com/OmDadhe" style="color:#00bcd4;">github.com/OmDadhe</a></td>
</tr>
<tr>
  <td style="padding:10px 18px; color:#8b949e; font-size:0.88em;"> Model</td>
  <td style="padding:10px 18px; color:#e6edf3;">XGBoost + Random Forest + SHAP · CV R² = 0.912</td>
  <td style="padding:10px 18px; color:#8b949e; font-size:0.88em;"> Scope</td>
  <td style="padding:10px 18px; color:#e6edf3;">36 Indian States &amp; UTs · 18 Alternative Signals</td>
</tr>
<tr style="background:#21262d;">
  <td style="padding:10px 18px; color:#8b949e; font-size:0.88em; vertical-align:top;"> Sources</td>
  <td colspan="3" style="padding:10px 18px; color:#e6edf3; font-size:0.9em;">RBI Table 154/153 · MSME Ministry Dashboard (31-03-2026) · SIDBI-Crisil MSME Pulse Jun 2025 · ICRIER Annual Survey 2025 · NITI Aayog MSME Report 2025 · RBI MCIR Jul 2025</td>
</tr>
</table>

<hr style="border:none; border-top:1px solid #30363d; margin:28px 0 22px;"/>

##  The Problem

India has **7.94 crore registered MSMEs**. They generate **35 crore jobs** and contribute **30% of GDP**. Yet **80% have never received a formal loan** — not because they are risky, but because traditional lenders **cannot see them**. No CIBIL score. No audited P&L. No collateral. The result: a **₹30 lakh crore credit gap** (SIDBI-Crisil 2025).

##  What This Notebook Builds

This notebook constructs India's first **state-level MSME Credit Invisibility Score** — an ML pipeline that predicts formal credit access using **18 alternative digital signals** (GST compliance, e-commerce integration, UPI adoption, banking infrastructure) instead of CIBIL data. It ranks all 36 states/UTs by credit exclusion severity, explains every prediction via SHAP, and generates publication-ready charts and an Excel dashboard.

##  Notebook Structure

| Section | Description |
|---------|-------------|
| **0 — Setup** | Imports, design system, path configuration |
| **1 — Data Loading** | Load processed datasets + RBI CD ratio (real data) |
| **2 — Modeling** | XGBoost, Random Forest, Gradient Boosting, Ridge, Ensemble + SHAP + PCA + K-Means |
| **3 — Visualizations** | 12 publication-grade charts |
| **4 — KPI Dashboard** | Summary PNG dashboard |
| **5 — Excel Dashboard** | 5-sheet professional workbook |
| **6 — Summary** | Final results and key findings |

<div style="margin-top:18px; padding:14px 18px; background:#161b22; border-left:4px solid #39d353; border-radius:4px; font-size:0.9em; color:#8b949e;">
 <strong style="color:#e6edf3;">Key Finding:</strong> Digital footprint signals (GST compliance behaviour, e-commerce integration, UPI adoption) explain <strong style="color:#39d353;">63.4% of variance</strong> in formal credit access across Indian states — a stronger signal than any traditional credit variable.
</div>

</div>

---
##  Section 0 — Imports, Design System & Path Configuration

All third-party libraries are imported here. The **design system** (`C` dict) defines the dark-mode colour palette used consistently across all 12 charts and the Excel dashboard. Global `matplotlib.rcParams` are set once so every chart inherits the same typography, background, and grid style without repetition.

> **Dependencies:** `pandas`, `numpy`, `matplotlib`, `seaborn`, `scipy`, `scikit-learn`, `xgboost`, `shap`, `openpyxl`
> Install: `pip install pandas numpy matplotlib seaborn scikit-learn xgboost shap openpyxl scipy`

In [ ]:
#!/usr/bin/env python3
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 0 — IMPORTS & CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import LinearSegmentedColormap, to_rgba
from matplotlib.ticker import FuncFormatter, MaxNLocator
import matplotlib.patheffects as pe
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import cross_val_score, KFold, LeaveOneOut
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import xgboost as xgb
import shap
from openpyxl import Workbook, load_workbook
from openpyxl.styles import (Font, PatternFill, Alignment, Border, Side,
                              GradientFill, numbers)
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule, DataBarRule
from openpyxl.chart import BarChart, Reference, LineChart, RadarChart
from openpyxl.chart.series import DataPoint
from openpyxl.drawing.image import Image as XLImage

warnings.filterwarnings('ignore')
print('✅ All libraries imported successfully')

✅ All libraries imported successfully


In [ ]:
import os

BASE   = os.getcwd()

DATA_P = os.path.join(BASE, 'data', 'processed')
DATA_R = os.path.join(BASE, 'data', 'raw')
OUT_C  = os.path.join(BASE, 'outputs', 'charts')
OUT_E  = os.path.join(BASE, 'outputs', 'excel')
OUT_R  = os.path.join(BASE, 'outputs', 'reports')

# ✅ Create ALL folders (data + outputs)
for p in [DATA_P, DATA_R, OUT_C, OUT_E, OUT_R]:
    os.makedirs(p, exist_ok=True)

print('✅ Full folder structure ready')

✅ Full folder structure ready


In [ ]:
import os
print(os.getcwd())

/content


In [ ]:
# ── DESIGN SYSTEM ─────────────────────────────────────────────────────────────
# Unified dark-mode colour palette. Every chart and Excel sheet draws from
# this single source of truth — change here to retheme the entire project.
C = {
    'bg':       '#0d1117',   # deep navy — chart background
    'surface':  '#161b22',   # axes panel background
    'panel':    '#21262d',   # secondary panel
    'border':   '#30363d',   # axis edges, grid lines
    'text':     '#e6edf3',   # primary text
    'muted':    '#8b949e',   # secondary / label text
    'lime':     '#39d353',   # success / low-risk accent
    'teal':     '#00bcd4',   # primary feature highlight
    'amber':    '#f0b429',   # medium-risk / secondary accent
    'coral':    '#ff6b6b',   # warning
    'purple':   '#7c3aed',   # model / cluster accent
    'blue':     '#2196f3',   # infrastructure signal
    'green':    '#4caf50',   # formalization signal
    'critical': '#ef4444',   # critical risk tier
    'high':     '#f59e0b',   # high risk tier
    'moderate': '#10b981',   # moderate risk tier
    'low':      '#3b82f6',   # low risk tier
    'white':    '#ffffff',
}

TIER_COLORS = {
    'Critical': C['critical'],
    'High':     C['high'],
    'Moderate': C['moderate'],
    'Low':      C['low'],
}

REGION_COLORS = {
    'North':     '#00bcd4',
    'South':     '#f0b429',
    'East':      '#ff6b6b',
    'West':      '#7c3aed',
    'Central':   '#39d353',
    'Northeast': '#e91e63',
}

# Apply globally — all charts inherit this theme automatically
plt.rcParams.update({
    'figure.facecolor':  C['bg'],
    'axes.facecolor':    C['surface'],
    'axes.edgecolor':    C['border'],
    'axes.labelcolor':   C['text'],
    'axes.titlecolor':   C['white'],
    'xtick.color':       C['muted'],
    'ytick.color':       C['muted'],
    'text.color':        C['text'],
    'grid.color':        C['border'],
    'grid.alpha':        0.5,
    'grid.linewidth':    0.5,
    'font.family':       'DejaVu Sans',
    'font.size':         10,
    'axes.titlesize':    13,
    'axes.labelsize':    10,
    'legend.facecolor':  C['panel'],
    'legend.edgecolor':  C['border'],
    'legend.labelcolor': C['text'],
})

print('✅ Design system and rcParams configured')
print(f'   Palette: {len(C)} colours | Risk tiers: {list(TIER_COLORS)} | Regions: {list(REGION_COLORS)}')

✅ Design system and rcParams configured
   Palette: 18 colours | Risk tiers: ['Critical', 'High', 'Moderate', 'Low'] | Regions: ['North', 'South', 'East', 'West', 'Central', 'Northeast']


In [ ]:
# ── CHART HELPER FUNCTIONS ────────────────────────────────────────────────────
# Reusable utilities used across all 12 charts.

def save_chart(fig, name, dpi=180):
    """Save figure to outputs/charts/ with consistent DPI and background."""
    path = os.path.join(OUT_C, f'{name}.png')
    fig.savefig(path, dpi=dpi, bbox_inches='tight',
                facecolor=C['bg'], edgecolor='none')
    plt.close(fig)
    size_kb = os.path.getsize(path) // 1024
    print(f'   ✓ {name}.png  ({size_kb} KB)')
    return path

def section_title(title, subtitle=''):
    """Print a formatted section banner to the notebook output."""
    bar = '═' * 70
    print(f'\n{bar}')
    print(f'  {title}')
    if subtitle:
        print(f'  {subtitle}')
    print(bar)

def styled_spine(ax):
    """Remove top/right spines and style remaining ones to match design system."""
    for spine in ax.spines.values():
        spine.set_edgecolor(C['border'])
        spine.set_linewidth(0.8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

def add_watermark(ax, text='Om Dadhe | MSME Credit Invisibility Score | 2026'):
    """Add a subtle author watermark to the bottom-right of a chart."""
    ax.text(0.99, 0.01, text, transform=ax.transAxes, ha='right', va='bottom',
            fontsize=6.5, color=C['border'], style='italic')

# Human-readable feature labels for chart axes and legends
FEAT_LABELS = {
    'Digital_Footprint_Score':  'Digital Footprint Score',
    'Ecomm_Integration_Pct':    'E-commerce Integration %',
    'Formalization_Score':      'Formalization Score',
    'NPA_Proxy_Pct':            'NPA Proxy %',
    'Internet_Penetration_Pct': 'Internet Penetration %',
    'GST_Compliance_Pct':       'GST Compliance Rate %',
    'Business_Resilience_Score':'Business Resilience Score',
    'GSDP_Per_Capita_Index':    'GSDP Per Capita (India=100)',
    'GST_Density':              'GST Taxpayer Density',
    'UPI_Adoption_Index':       'UPI Adoption Index',
    'Banking_Access_Score':     'Banking Access Score',
    'Credit_Momentum_Score':    'Credit Momentum Score',
    'Bank_Branches_Per_Lakh':   'Bank Branches / Lakh Pop',
    'Literacy_Rate':            'Literacy Rate %',
    'Urban_Pct':                'Urban Population %',
    'Women_MSME_Pct':           'Women-owned MSME %',
    'Trading_Pct':              'Trading Sector %',
    'CD_Ratio_2025':            'C-D Ratio 2025 (%)',
}

print('✅ Helper functions registered')

✅ Helper functions registered


---
## 📂 Section 1 — Data Loading & Validation

Three datasets are loaded in this section:

1. **`scored_dataset.csv`** — Master table of 36 states × 35 features including the precomputed `Credit_Invisibility_Score` and `Risk_Tier` labels. This is the primary modelling input.
2. **`shap_importance.csv`** — Pre-computed SHAP feature importance rankings used in Chart 2.
3. **`RBI_CD_Ratio_Table154.xlsx`** — Real RBI Credit-Deposit ratio data (2004–2025), parsed from two sheets covering Place of Utilisation.

A data quality report is printed: row/feature counts, missing value audit, and key national statistics directly from the MSME Ministry Dashboard (31-03-2026).

> ⚠️ **Prerequisites:** Ensure `data/processed/scored_dataset.csv`, `data/processed/shap_importance.csv`, and `data/raw/RBI_CD_Ratio_Table154.xlsx` are present before running.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — DATA LOADING & VALIDATION
# ─────────────────────────────────────────────────────────────────────────────
section_title('SECTION 1 — DATA LOADING & VALIDATION')

# Load the two core processed datasets
df       = pd.read_csv(os.path.join(DATA_P, 'scored_dataset.csv'))
shap_df  = pd.read_csv(os.path.join(DATA_P, 'shap_importance.csv'))

print(f'\n✅ Loaded scored_dataset.csv     — {df.shape[0]} rows × {df.shape[1]} columns')
print(f'✅ Loaded shap_importance.csv    — {shap_df.shape[0]} features ranked')


══════════════════════════════════════════════════════════════════════
  SECTION 1 — DATA LOADING & VALIDATION
══════════════════════════════════════════════════════════════════════

✅ Loaded scored_dataset.csv     — 36 rows × 35 columns
✅ Loaded shap_importance.csv    — 18 features ranked


In [ ]:
# ── Load real RBI CD Ratio data (Table 154 — Place of Utilisation) ─────────
# The RBI Excel file has two sheets, each with a header row at index 4
# containing years. State rows are parsed, filtering out region headers,
# footnotes, and source lines.

def load_rbi_cd(path, years_col_offset=2):
    """Parse RBI CD ratio Excel file (2-sheet format) into a clean DataFrame."""
    raw1 = pd.read_excel(path, sheet_name=0, header=None)
    raw2 = pd.read_excel(path, sheet_name=1, header=None)

    states_regions = [
        'NORTHERN REGION', 'NORTH-EASTERN REGION', 'EASTERN REGION',
        'CENTRAL REGION',  'WESTERN REGION',       'SOUTHERN REGION'
    ]

    def parse_sheet(raw):
        year_row = raw.iloc[4]
        years = []
        for v in year_row[2:]:
            try:
                y = int(float(v))
                if 2000 <= y <= 2030:
                    years.append(y)
            except:
                pass

        rows = []
        for _, row in raw.iterrows():
            state = row.iloc[1]
            if not isinstance(state, str):
                continue
            skip_tokens = [
                *states_regions, '(', 'Source', '-', '*'
            ]
            if any(state.strip().startswith(t) or state.strip() == t
                   for t in skip_tokens):
                continue
            try:
                vals = []
                for v in row.iloc[2:2 + len(years)]:
                    try:
                        vals.append(float(v))
                    except:
                        vals.append(np.nan)
                if any(not np.isnan(v) for v in vals):
                    rows.append({'State': state.strip()} | dict(zip(years, vals)))
            except:
                pass
        return pd.DataFrame(rows)

    d1 = parse_sheet(raw1)
    d2 = parse_sheet(raw2)
    return pd.concat([d1, d2], ignore_index=True).drop_duplicates('State')

rbi_raw = load_rbi_cd(os.path.join(DATA_R, 'RBI_CD_Ratio_Table154.xlsx'))
rbi_years = [c for c in rbi_raw.columns if isinstance(c, int)]
print(f'\n✅ RBI CD Ratio data loaded — {rbi_raw.shape[0]} states, years: {rbi_years}')


✅ RBI CD Ratio data loaded — 39 states, years: [2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [ ]:
# ── Data Quality Report ───────────────────────────────────────────────────────
print('\n MASTER DATASET QUALITY REPORT')
print(f'   Rows (States/UTs)  : {df.shape[0]}')
print(f'   Feature columns    : {df.shape[1]}')
print(f'   Missing values     : {df.isnull().sum().sum()}')
print(f'   Risk tier dist     : {df["Risk_Tier"].value_counts().to_dict()}')
print(f'   Regions covered    : {sorted(df["Region"].unique().tolist())}')

print('\n   KEY NATIONAL METRICS (MSME Ministry Dashboard, 31-03-2026):')
print('   Total MSMEs        : 7.94 crore')
print('   Micro %            : 99.3%')
print('   Women-owned %      : 39.2%')
print('   Trading %          : 42.7%')
print('   CGTMSE Guarantees  : ₹13.17 lakh crore')
print('   Credit Gap         : ₹30 lakh crore  (SIDBI-Crisil 2025)')
print('   RBI FI-Index       : 67.0  (Mar 2025)')
print('   Digital Pay Index  : 493.22  (Mar 2025)')

# Derived stats used in downstream charts
total_invisible = df['Alt_Credit_Invisible_Lakh'].sum()
critical_states = df[df['Risk_Tier'] == 'Critical']['State'].count()
top_invisible   = df.nlargest(5, 'Alt_Credit_Invisible_Lakh')[['State', 'Alt_Credit_Invisible_Lakh']].values

print('\n   MODEL OUTPUTS:')
print(f'   Credit-invisible MSMEs : {total_invisible:.1f} lakh')
print(f'   Critical-tier states   : {critical_states}/36')
print('   Top 5 by invisible count:')
for s, v in top_invisible:
    print(f'     {s:<22} {v:.1f} lakh')


 MASTER DATASET QUALITY REPORT
   Rows (States/UTs)  : 36
   Feature columns    : 35
   Missing values     : 0
   Risk tier dist     : {'Critical': 21, 'High': 13, 'Moderate': 2}
   Regions covered    : ['Central', 'East', 'North', 'Northeast', 'South', 'West']

   KEY NATIONAL METRICS (MSME Ministry Dashboard, 31-03-2026):
   Total MSMEs        : 7.94 crore
   Micro %            : 99.3%
   Women-owned %      : 39.2%
   Trading %          : 42.7%
   CGTMSE Guarantees  : ₹13.17 lakh crore
   Credit Gap         : ₹30 lakh crore  (SIDBI-Crisil 2025)
   RBI FI-Index       : 67.0  (Mar 2025)
   Digital Pay Index  : 493.22  (Mar 2025)

   MODEL OUTPUTS:
   Credit-invisible MSMEs : 369.6 lakh
   Critical-tier states   : 21/36
   Top 5 by invisible count:
     Maharashtra            48.9 lakh
     Uttar Pradesh          42.8 lakh
     Tamil Nadu             30.6 lakh
     Bihar                  25.3 lakh
     West Bengal            24.9 lakh


---
##  Section 2 — Machine Learning Pipeline

This section trains **four independent models** on 18 alternative credit signals and combines them into a weighted ensemble:

| Model | Role | CV R² |
|-------|------|-------|
| **XGBoost** | Primary model — captures non-linear feature interactions | 0.912 |
| **Random Forest** | Variance reduction — robust to outliers | 0.891 |
| **Gradient Boosting** | Sequential signal — good with noisy data | 0.893 |
| **Ridge (Baseline)** | Linear comparison — fully interpretable | 0.944 |
| **Ensemble (45/35/20)** | Final scoring output — weighted average | — |

After training, **SHAP TreeExplainer** decomposes each XGBoost prediction into per-feature contributions, making the score fully transparent. **PCA** reduces the 18-feature space to 4 components for visualisation. **K-Means** (k=4) identifies distinct MSME ecosystem archetypes.

> All models use **5-fold cross-validation** on n=36 (states), with standardised features for gradient-based models.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — ADVANCED MODELING
# ─────────────────────────────────────────────────────────────────────────────
section_title('SECTION 2 — ADVANCED MODELING')

# ── Feature & Target Definition ───────────────────────────────────────────────
# 18 alternative credit signals across 5 signal pillars:
#   • Digital Behaviour    (GST, e-comm, UPI)
#   • Banking Access       (branches, CD ratio, composite score)
#   • Formalization        (GST density, women MSME, trading %)
#   • Macro-Resilience     (GSDP index, literacy, urban %)
#   • Credit Quality       (NPA proxy, credit momentum)

FEATURES = [
    'Digital_Footprint_Score',   # Composite digital behaviour score
    'Banking_Access_Score',      # Composite banking infrastructure score
    'Formalization_Score',       # Composite formalization score
    'Business_Resilience_Score', # Macro environment composite
    'Credit_Momentum_Score',     # CD ratio trend 2004–2025
    'GST_Compliance_Pct',        # % of registered GST filers who are compliant
    'Ecomm_Integration_Pct',     # % of state MSMEs on e-commerce platforms
    'Internet_Penetration_Pct',  # State internet penetration %
    'UPI_Adoption_Index',        # Indexed UPI transaction density
    'Bank_Branches_Per_Lakh',    # Bank branches per lakh population
    'CD_Ratio_2025',             # Credit-Deposit ratio 2025 (real RBI data)
    'GST_Density',               # GST taxpayers per lakh MSME
    'Literacy_Rate',             # State literacy rate %
    'Urban_Pct',                 # Urban population %
    'GSDP_Per_Capita_Index',     # GSDP per capita (India = 100)
    'Women_MSME_Pct',            # Women-owned MSME % (MSME Ministry data)
    'Trading_Pct',               # Trading sector MSME % (MSME Ministry data)
    'NPA_Proxy_Pct',             # Non-performing asset proxy %
]

TARGET = 'Formal_Credit_Access_Pct'
X = df[FEATURES].values
y = df[TARGET].values

# Standardise for gradient-based and linear models; XGBoost/RF use raw values
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

kf  = KFold(n_splits=5, shuffle=True, random_state=42)
loo = LeaveOneOut()

print(f'Features matrix  : {X.shape[0]} states × {X.shape[1]} signals')
print(f'Target variable  : {TARGET}')
print(f'Target range     : {y.min():.1f}% – {y.max():.1f}%')
print(f'CV strategy      : KFold(n_splits=5, shuffle=True, random_state=42)')


══════════════════════════════════════════════════════════════════════
  SECTION 2 — ADVANCED MODELING
══════════════════════════════════════════════════════════════════════
Features matrix  : 36 states × 18 signals
Target variable  : Formal_Credit_Access_Pct
Target range     : 8.2% – 42.6%
CV strategy      : KFold(n_splits=5, shuffle=True, random_state=42)


In [ ]:
# ── Model Training ─────────────────────────────────────────────────────────────

# Model 1: XGBoost — primary model
xgb_model = xgb.XGBRegressor(
    n_estimators=300, max_depth=4, learning_rate=0.04,
    subsample=0.8, colsample_bytree=0.75, min_child_weight=2,
    reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbosity=0
)
xgb_model.fit(X, y)
xgb_cv   = cross_val_score(xgb_model, X, y, cv=kf, scoring='r2')
xgb_pred = xgb_model.predict(X)

# Model 2: Random Forest — ensemble variance reduction
rf_model = RandomForestRegressor(
    n_estimators=500, max_depth=6, min_samples_leaf=2,
    max_features=0.7, random_state=42, n_jobs=-1
)
rf_model.fit(X, y)
rf_cv   = cross_val_score(rf_model, X, y, cv=kf, scoring='r2')
rf_pred = rf_model.predict(X)

# Model 3: Gradient Boosting — sequential signal learning
gb_model = GradientBoostingRegressor(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, random_state=42
)
gb_model.fit(X_scaled, y)
gb_cv   = cross_val_score(gb_model, X_scaled, y, cv=kf, scoring='r2')
gb_pred = gb_model.predict(X_scaled)

# Model 4: Ridge — linear baseline
ridge_model = Ridge(alpha=10.0)
ridge_model.fit(X_scaled, y)
ridge_cv   = cross_val_score(ridge_model, X_scaled, y, cv=kf, scoring='r2')
ridge_pred = ridge_model.predict(X_scaled)

# Ensemble: weighted average (45% XGB + 35% RF + 20% GB)
ensemble_pred = 0.45 * xgb_pred + 0.35 * rf_pred + 0.20 * gb_pred

model_results = {
    'XGBoost':          {'cv': xgb_cv,   'pred': xgb_pred,      'color': C['teal']},
    'Random Forest':    {'cv': rf_cv,    'pred': rf_pred,       'color': C['lime']},
    'Gradient Boost':   {'cv': gb_cv,    'pred': gb_pred,       'color': C['amber']},
    'Ridge (baseline)': {'cv': ridge_cv, 'pred': ridge_pred,    'color': C['muted']},
    'Ensemble':         {'cv': None,     'pred': ensemble_pred,  'color': C['coral']},
}

print(f"\n{'Model':<22} {'CV R² Mean':>12} {'CV R² Std':>10} {'Train R²':>10} {'MAE':>8}")
print('─' * 65)
for name, res in model_results.items():
    pred = res['pred']
    r2   = r2_score(y, pred)
    mae  = mean_absolute_error(y, pred)
    if res['cv'] is not None:
        print(f"{name:<22} {res['cv'].mean():>12.3f} {res['cv'].std():>10.3f} {r2:>10.3f} {mae:>8.2f}")
    else:
        print(f"{'Ensemble (weighted)':<22} {'—':>12} {'—':>10} {r2:>10.3f} {mae:>8.2f}")


Model                    CV R² Mean  CV R² Std   Train R²      MAE
─────────────────────────────────────────────────────────────────
XGBoost                       0.912      0.059      1.000     0.05
Random Forest                 0.891      0.051      0.976     0.74
Gradient Boost                0.893      0.090      1.000     0.00
Ridge (baseline)              0.944      0.054      0.978     1.00
Ensemble (weighted)               —          —      0.997     0.28


In [ ]:
# ── SHAP Explainability (XGBoost) ─────────────────────────────────────────────
# TreeExplainer computes exact SHAP values for tree-based models.
# shap_mean = mean |SHAP value| per feature = global feature importance.

explainer = shap.TreeExplainer(xgb_model)
shap_vals = explainer.shap_values(X)
shap_mean = np.abs(shap_vals).mean(axis=0)

print('\nSHAP Global Feature Importance (XGBoost — Mean |SHAP|):')
feat_order = np.argsort(shap_mean)[::-1]
for i in feat_order:
    print(f'   {FEAT_LABELS.get(FEATURES[i], FEATURES[i]):<35} {shap_mean[i]:.4f}')


SHAP Global Feature Importance (XGBoost — Mean |SHAP|):
   Digital Footprint Score             3.4182
   E-commerce Integration %            1.2991
   Formalization Score                 0.7444
   GST Compliance Rate %               0.6158
   NPA Proxy %                         0.5017
   Internet Penetration %              0.4048
   Business Resilience Score           0.3204
   GSDP Per Capita (India=100)         0.2461
   GST Taxpayer Density                0.1781
   Credit Momentum Score               0.1448
   UPI Adoption Index                  0.1253
   Trading Sector %                    0.1108
   Bank Branches / Lakh Pop            0.0576
   C-D Ratio 2025 (%)                  0.0508
   Urban Population %                  0.0507
   Literacy Rate %                     0.0463
   Banking Access Score                0.0462
   Women-owned MSME %                  0.0398


In [ ]:
# ── PCA — Dimensionality Reduction for Cluster Visualisation ─────────────────
# Reduce 18 features → 4 principal components.
# PC1 and PC2 are used to plot the K-Means cluster projection (Chart 8).

pca   = PCA(n_components=4, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print('PCA — Variance explained by 4 components:')
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f'   PC{i+1}: {v*100:.1f}%  (cumulative: {pca.explained_variance_ratio_[:i+1].sum()*100:.1f}%)')

# ── K-Means Clustering ────────────────────────────────────────────────────────
# Identifies 4 MSME ecosystem archetypes based on the full 18-feature space.
# Cluster names are assigned based on the dominant signal profile of each group.

km = KMeans(n_clusters=4, random_state=42, n_init=20)
df['Cluster'] = km.fit_predict(X_scaled)

CLUSTER_NAMES = {
    0: 'Digitally Advanced',
    1: 'Infrastructure Laggard',
    2: 'Banking Accessible',
    3: 'Systemically Excluded'
}
df['Cluster_Name'] = df['Cluster'].map(CLUSTER_NAMES)

cluster_summary = df.groupby('Cluster_Name')[[
    'Formal_Credit_Access_Pct', 'GST_Compliance_Pct',
    'Ecomm_Integration_Pct', 'Bank_Branches_Per_Lakh',
    'Credit_Invisibility_Score'
]].mean().round(1)
print('\nK-Means Cluster Summary (mean values):')
print(cluster_summary.to_string())

# Store predictions for chart annotations
df['XGBoost_Pred']  = xgb_pred
df['RF_Pred']       = rf_pred
df['Ensemble_Pred'] = ensemble_pred
df['SHAP_0']        = shap_vals[:, 0]   # Digital Footprint SHAP
df['SHAP_1']        = shap_vals[:, 1]   # E-commerce SHAP

print('\n✅ All models trained and validated')

PCA — Variance explained by 4 components:
   PC1: 63.9%  (cumulative: 63.9%)
   PC2: 12.3%  (cumulative: 76.2%)
   PC3: 8.3%  (cumulative: 84.6%)
   PC4: 5.1%  (cumulative: 89.6%)

K-Means Cluster Summary (mean values):
                        Formal_Credit_Access_Pct  GST_Compliance_Pct  Ecomm_Integration_Pct  Bank_Branches_Per_Lakh  Credit_Invisibility_Score
Cluster_Name                                                                                                                                  
Banking Accessible                          11.2                70.5                    8.0                    10.4                       88.8
Digitally Advanced                          18.4                81.1                   14.3                    16.1                       81.6
Infrastructure Laggard                      27.2                89.2                   24.8                    21.2                       72.8
Systemically Excluded                       40.5                9

---
##  Section 3 — Publication-Grade Visualizations (12 Charts)

All charts share the dark-mode design system defined in Section 0. Each chart is saved to `outputs/charts/` at 180 DPI in PNG format.

| Chart | Title | Key Insight |
|-------|-------|-------------|
| 01 | Credit Invisibility Ranking | 21/36 states in Critical tier |
| 02 | SHAP Feature Importance | Digital behaviour = 63.4% of model |
| 03 | RBI CD Ratio Trend 2004–2025 | Northeast consistently below 40% |
| 04 | Model Comparison (Actual vs Predicted) | XGBoost CV R² = 0.912 |
| 05 | Cross-Validation Robustness | All models above 0.70 threshold |
| 06 | Correlation Heatmap | E-comm r=0.89, GST r=0.87 with credit |
| 07 | Bubble Chart: Invisible MSMEs × CD × Risk | Bihar + UP = 68L invisible MSMEs |
| 08 | K-Means Cluster PCA Projection | 4 distinct ecosystem archetypes |
| 09 | Regional Distribution Boxplot | South best, Northeast worst |
| 10 | E-commerce & GST → Credit Access | OLS regression proof |
| 11 | Invisible MSME Waterfall by Region | Central India = largest gap |
| 12 | Signal Heatmap — 15 Vulnerable States | Red = weak, Green = strong |


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CHART 1 — CREDIT INVISIBILITY SCORE RANKING (All 36 States & UTs)
# Purpose : Rank all states by Credit Invisibility Score with risk-tier colouring.
# Insight : 21 of 36 states score ≥ 80 (Critical tier) — Northeast worst at 87–92.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('  Rendering Chart 1: Credit Invisibility Ranking...')

sorted_df = df.sort_values('Credit_Invisibility_Score', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 13), facecolor=C['bg'])
ax.set_facecolor(C['bg'])

bar_colors = [TIER_COLORS[t] for t in sorted_df['Risk_Tier']]
bars = ax.barh(range(len(sorted_df)), sorted_df['Credit_Invisibility_Score'],
               color=bar_colors, height=0.72, zorder=3, alpha=0.9)

# Alternating row background for readability
for i in range(len(sorted_df)):
    ax.axhspan(i - 0.5, i + 0.5,
               color=C['surface'] if i % 2 == 0 else C['bg'],
               alpha=0.5, zorder=1)

# Score and state labels
for i, (bar, row) in enumerate(zip(bars, sorted_df.itertuples())):
    w = bar.get_width()
    ax.text(w + 0.4, i, f'{w:.1f}', va='center', ha='left',
            fontsize=8.5, color=TIER_COLORS[row.Risk_Tier], fontweight='bold')
    ax.text(-0.5, i, row.State, va='center', ha='right', fontsize=8.5, color=C['text'])

# Risk-tier threshold lines
ax.axvline(80, color=C['critical'], lw=1.2, linestyle='--', alpha=0.7, zorder=4)
ax.axvline(65, color=C['high'],     lw=1.2, linestyle='--', alpha=0.7, zorder=4)
ax.text(80.5, -1, 'Critical →', color=C['critical'], fontsize=8, va='top')
ax.text(65.5, -1, 'High →',     color=C['high'],     fontsize=8, va='top')

patches = [mpatches.Patch(color=TIER_COLORS[t], label=f'{t} Risk')
           for t in ['Critical', 'High', 'Moderate']]
ax.legend(handles=patches, loc='lower right', framealpha=0.2, fontsize=9)

ax.set_yticks([])
ax.set_xlim(0, 105)
ax.set_xlabel('Credit Invisibility Score  (0 = fully visible → 100 = fully excluded)',
              labelpad=10, fontsize=11)
ax.set_title('MSME Credit Invisibility Score — All 36 States & UTs',
             pad=18, fontsize=15, fontweight='bold', color=C['white'])
ax.text(0.5, 0.97,
        'Alt-signal XGBoost model | Higher score = greater credit exclusion | Source: RBI, MSME Ministry, SIDBI 2025',
        transform=ax.transAxes, ha='center', va='top', fontsize=8, color=C['muted'])
ax.grid(axis='x', alpha=0.2, zorder=0)
for sp in ax.spines.values():
    sp.set_visible(False)
add_watermark(ax)

save_chart(fig, 'chart01_credit_invisibility_ranking')

  Rendering Chart 1: Credit Invisibility Ranking...
   ✓ chart01_credit_invisibility_ranking.png  (242 KB)


'/content/outputs/charts/chart01_credit_invisibility_ranking.png'

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CHART 2 — SHAP FEATURE IMPORTANCE + SIGNAL CATEGORY BREAKDOWN
# Purpose : Show which features drive the model and group them by signal pillar.
# Insight : Digital Behaviour accounts for 63.4% of total SHAP importance.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('  Rendering Chart 2: SHAP Feature Importance...')

feat_order = np.argsort(shap_mean)[::-1]
top_n = 12

fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor=C['bg'])
fig.subplots_adjust(wspace=0.35)

# Left panel: SHAP bar chart (top 12 features)
ax = axes[0]
ax.set_facecolor(C['surface'])
colors_shap = plt.cm.RdYlGn(np.linspace(0.2, 0.85, top_n))[::-1]
feat_names  = [FEAT_LABELS.get(FEATURES[i], FEATURES[i]) for i in feat_order[:top_n]][::-1]
shap_vals_  = shap_mean[feat_order[:top_n]][::-1]

bars = ax.barh(range(top_n), shap_vals_, color=colors_shap, height=0.65, edgecolor='none', zorder=3)
for i, (bar, v) in enumerate(zip(bars, shap_vals_)):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=8.5, color=C['text'], fontweight='bold')

ax.set_yticks(range(top_n))
ax.set_yticklabels(feat_names, fontsize=9)
ax.set_xlabel('Mean |SHAP Value|', labelpad=8)
ax.set_title('Feature Importance (SHAP)\nMean absolute impact on credit access prediction',
             fontsize=11, pad=12, color=C['white'])
ax.grid(axis='x', alpha=0.25, zorder=0)
styled_spine(ax)

# Right panel: Signal category contribution %
ax2 = axes[1]
ax2.set_facecolor(C['surface'])

signal_cats = {
    'Digital Behaviour': ['Digital_Footprint_Score', 'Ecomm_Integration_Pct', 'UPI_Adoption_Index'],
    'Tax Compliance':    ['GST_Compliance_Pct', 'GST_Density'],
    'Infrastructure':    ['Internet_Penetration_Pct', 'Bank_Branches_Per_Lakh', 'Banking_Access_Score'],
    'Macro-Resilience':  ['GSDP_Per_Capita_Index', 'Literacy_Rate', 'Urban_Pct'],
    'Credit Quality':    ['NPA_Proxy_Pct', 'Credit_Momentum_Score', 'CD_Ratio_2025'],
    'Formalization':     ['Formalization_Score', 'Women_MSME_Pct', 'Trading_Pct'],
}
cat_colors = [C['teal'], C['lime'], C['amber'], C['purple'], C['coral'], C['blue']]

cat_totals = {}
for cat, feats in signal_cats.items():
    cat_totals[cat] = sum(shap_mean[FEATURES.index(f)] for f in feats if f in FEATURES)

total_sum = sum(cat_totals.values())
cats   = list(cat_totals.keys())
shares = [cat_totals[c] / total_sum * 100 for c in cats]
order  = np.argsort(shares)

bars2 = ax2.barh([cats[i] for i in order], [shares[i] for i in order],
                 color=[cat_colors[i % len(cat_colors)] for i in order],
                 height=0.6, edgecolor='none', zorder=3)
for bar, v in zip(bars2, sorted(shares)):
    ax2.text(v + 0.3, bar.get_y() + bar.get_height()/2,
             f'{v:.1f}%', va='center', fontsize=9, fontweight='bold', color=C['text'])

ax2.set_xlabel('% Contribution to Model', labelpad=8)
ax2.set_title('Signal Category Contribution\n% of total SHAP importance explained',
              fontsize=11, pad=12, color=C['white'])
ax2.grid(axis='x', alpha=0.25, zorder=0)
ax2.set_xlim(0, max(shares) + 8)
styled_spine(ax2)

fig.suptitle('SHAP Explainability Dashboard — What Predicts MSME Credit Access?',
             fontsize=14, fontweight='bold', color=C['white'], y=1.01)
add_watermark(axes[1])
save_chart(fig, 'chart02_shap_importance')

  Rendering Chart 2: SHAP Feature Importance...
   ✓ chart02_shap_importance.png  (209 KB)


'/content/outputs/charts/chart02_shap_importance.png'

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CHART 3 — RBI CD RATIO TREND 2004–2025 (Real Uploaded Data) — 6-panel
# Purpose : Show 20-year banking penetration trajectory by region.
# Insight : Northeast region has been structurally below 40% for two decades.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('  Rendering Chart 3: RBI CD Ratio Trend (2004–2025)...')

years_all    = [c for c in rbi_raw.columns if isinstance(c, int) and 2004 <= c <= 2025]
state_region = df[['State', 'Region']].set_index('State')['Region'].to_dict()

fig, axes = plt.subplots(2, 3, figsize=(18, 10), facecolor=C['bg'])
fig.subplots_adjust(hspace=0.45, wspace=0.32)

region_order = ['North', 'South', 'East', 'West', 'Central', 'Northeast']

for ax_idx, (region, ax) in enumerate(zip(region_order, axes.flat)):
    ax.set_facecolor(C['surface'])
    region_states = [s for s, r in state_region.items() if r == region]
    plotted = 0

    for state in region_states:
        row = rbi_raw[rbi_raw['State'] == state]
        if row.empty:
            continue
        vals        = [row[y].values[0] if y in row.columns else np.nan for y in years_all]
        valid_pairs = [(y, v) for y, v in zip(years_all, vals)
                       if not (isinstance(v, (float, int)) and np.isnan(float(v)))]
        if len(valid_pairs) <= 3:
            continue
        valid_years = [p[0] for p in valid_pairs]
        valid_vals  = [p[1] for p in valid_pairs]
        alpha = 0.85 if len(region_states) <= 4 else 0.65
        lw    = 1.8  if len(region_states) <= 4 else 1.2
        ax.plot(valid_years, valid_vals,
                marker='o', markersize=3, linewidth=lw, alpha=alpha,
                color=REGION_COLORS[region], zorder=3)
        ax.text(valid_years[-1] + 0.2, valid_vals[-1],
                state.split()[-1][:8], fontsize=6.5, color=REGION_COLORS[region], va='center')
        plotted += 1

    ax.axhline(60,  color=C['muted'], lw=0.9, linestyle='--', alpha=0.6)
    ax.axhline(100, color=C['amber'], lw=0.9, linestyle=':',  alpha=0.5)
    ax.text(2004.5, 62, 'Min adequate', fontsize=6.5, color=C['muted'])
    ax.set_title(f'{region} Region  ({plotted} states)',
                 fontsize=10, fontweight='bold', color=REGION_COLORS[region], pad=8)
    ax.set_xlabel('Year', fontsize=8, labelpad=5)
    ax.set_ylabel('CD Ratio (%)', fontsize=8)
    ax.grid(alpha=0.2)
    styled_spine(ax)
    ax.tick_params(labelsize=7.5)

fig.suptitle('State-wise Credit-Deposit Ratio Trend 2004–2025\n'
             'Source: RBI Table 154 — Place of Utilisation  (real uploaded data)',
             fontsize=14, fontweight='bold', color=C['white'], y=1.01)
add_watermark(axes[1][2])
save_chart(fig, 'chart03_rbi_cd_trend')

  Rendering Chart 3: RBI CD Ratio Trend (2004–2025)...
   ✓ chart03_rbi_cd_trend.png  (526 KB)


'/content/outputs/charts/chart03_rbi_cd_trend.png'

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CHART 4 — MODEL COMPARISON: ACTUAL vs PREDICTED (all 4 models)
# Purpose : Visualise how well each model predicts formal credit access.
# Insight : XGBoost and Ridge both achieve CV R² > 0.90; scatter tight around diagonal.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('  Rendering Chart 4: Model Comparison (Actual vs Predicted)...')

fig, axes = plt.subplots(2, 2, figsize=(14, 11), facecolor=C['bg'])
fig.subplots_adjust(hspace=0.4, wspace=0.3)

models_plot = [
    ('XGBoost',          xgb_pred,   xgb_cv,   C['teal']),
    ('Random Forest',    rf_pred,    rf_cv,    C['lime']),
    ('Gradient Boosting',gb_pred,    gb_cv,    C['amber']),
    ('Ridge (Baseline)', ridge_pred, ridge_cv, C['muted']),
]

for ax, (name, pred, cv, color) in zip(axes.flat, models_plot):
    ax.set_facecolor(C['surface'])
    r2  = r2_score(y, pred)
    mae = mean_absolute_error(y, pred)

    tier_c = [TIER_COLORS[t] for t in df['Risk_Tier']]
    ax.scatter(y, pred, c=tier_c, s=60, alpha=0.8,
               edgecolors=C['border'], linewidth=0.5, zorder=3)

    lim_min = min(y.min(), pred.min()) - 1
    lim_max = max(y.max(), pred.max()) + 1
    ax.plot([lim_min, lim_max], [lim_min, lim_max],
            color=C['border'], lw=1.2, linestyle='--', alpha=0.7, zorder=2)

    z = np.polyfit(y, pred, 1)
    p_fit = np.poly1d(z)
    x_line = np.linspace(lim_min, lim_max, 100)
    ax.plot(x_line, p_fit(x_line), color=color, lw=2, alpha=0.6, zorder=4)

    # Annotate large-MSME states
    for _, row in df.iterrows():
        if row['Total_MSMEs_Lakh'] > 25:
            pred_val = pred[df.index.get_loc(row.name)]
            ax.annotate(row['State'].split()[-1],
                        (row['Formal_Credit_Access_Pct'], pred_val),
                        xytext=(4, 3), textcoords='offset points',
                        fontsize=7, color=color, alpha=0.8)

    ax.set_title(name, fontsize=12, fontweight='bold', color=color, pad=10)
    ax.text(0.05, 0.92,
            f'CV R² = {cv.mean():.3f} ± {cv.std():.3f}\nTrain R² = {r2:.3f} | MAE = {mae:.1f}',
            transform=ax.transAxes, fontsize=8.5, color=C['text'],
            bbox=dict(boxstyle='round,pad=0.4', facecolor=C['panel'], alpha=0.8))

    ax.set_xlabel('Actual Formal Credit Access %', fontsize=9)
    ax.set_ylabel('Predicted Formal Credit Access %', fontsize=9)
    ax.set_xlim(lim_min, lim_max)
    ax.set_ylim(lim_min, lim_max)
    ax.grid(alpha=0.2)
    styled_spine(ax)

patches = [mpatches.Patch(color=TIER_COLORS[t], label=f'{t} Risk')
           for t in ['Critical', 'High', 'Moderate']]
fig.legend(handles=patches, loc='upper center', ncol=3, framealpha=0.2, fontsize=9,
           bbox_to_anchor=(0.5, 1.01))
fig.suptitle('Model Comparison: Actual vs Predicted Formal Credit Access\n'
             '5-Fold Cross-Validated | Diagonal = perfect prediction',
             fontsize=14, fontweight='bold', color=C['white'], y=1.05)
add_watermark(axes[1][1])
save_chart(fig, 'chart04_model_comparison')

  Rendering Chart 4: Model Comparison (Actual vs Predicted)...
   ✓ chart04_model_comparison.png  (351 KB)


'/content/outputs/charts/chart04_model_comparison.png'

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CHART 5 — CROSS-VALIDATION SCORE DISTRIBUTION (Robustness Check)
# Purpose : Validate model stability — are scores consistent across folds?
# Insight : All four models are above the 0.70 good-threshold, tight variance.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('  Rendering Chart 5: Cross-Validation Robustness...')

fig, ax = plt.subplots(figsize=(12, 6), facecolor=C['bg'])
ax.set_facecolor(C['surface'])

cv_data   = [xgb_cv, rf_cv, gb_cv, ridge_cv]
cv_names  = ['XGBoost', 'Random Forest', 'Gradient Boosting', 'Ridge (Baseline)']
cv_colors = [C['teal'], C['lime'], C['amber'], C['muted']]

positions = np.array([1, 2, 3, 4])
bp = ax.boxplot(cv_data, positions=positions, patch_artist=True, widths=0.45,
                medianprops=dict(color=C['white'], linewidth=2.5),
                whiskerprops=dict(color=C['muted'], linewidth=1.2),
                capprops=dict(color=C['muted'], linewidth=1.2),
                flierprops=dict(marker='D', color=C['coral'], markersize=5, alpha=0.7))

for patch, color in zip(bp['boxes'], cv_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
    patch.set_edgecolor(C['border'])

# Overlay individual fold scores
for i, (cv_scores, pos) in enumerate(zip(cv_data, positions)):
    jitter = np.random.uniform(-0.06, 0.06, len(cv_scores))
    ax.scatter(pos + jitter, cv_scores, color=cv_colors[i], s=50, zorder=5,
               edgecolors=C['white'], linewidth=0.5, alpha=0.9)
    ax.text(pos, cv_scores.mean() + 0.005, f'μ={cv_scores.mean():.3f}',
            ha='center', fontsize=9, color=cv_colors[i], fontweight='bold')

ax.axhline(0.7, color=C['lime'], lw=1.2, linestyle='--', alpha=0.6)
ax.text(4.45, 0.71, 'Good threshold (0.70)', color=C['lime'], fontsize=8, va='bottom')

ax.set_xticks(positions)
ax.set_xticklabels(cv_names, fontsize=10)
ax.set_ylabel('5-Fold Cross-Validated R² Score', fontsize=10, labelpad=8)
ax.set_title('Model Robustness: Cross-Validation Score Distribution\n'
             'Higher & tighter = better generalization on unseen states',
             fontsize=13, fontweight='bold', color=C['white'], pad=12)
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.25)
styled_spine(ax)
add_watermark(ax)
save_chart(fig, 'chart05_cv_robustness')

  Rendering Chart 5: Cross-Validation Robustness...
   ✓ chart05_cv_robustness.png  (103 KB)


'/content/outputs/charts/chart05_cv_robustness.png'

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CHART 6 — CORRELATION HEATMAP (14 variables, publication-grade)
# Purpose : Show pairwise correlations between all key variables.
# Insight : E-commerce (r=0.89) and GST (r=0.87) have highest correlation with
#           formal credit access — stronger than GSDP or literacy.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('  Rendering Chart 6: Correlation Heatmap...')

corr_cols = [
    'Formal_Credit_Access_Pct', 'GST_Compliance_Pct', 'Ecomm_Integration_Pct',
    'UPI_Adoption_Index', 'Internet_Penetration_Pct', 'Bank_Branches_Per_Lakh',
    'CD_Ratio_2025', 'Literacy_Rate', 'Urban_Pct', 'GSDP_Per_Capita_Index',
    'NPA_Proxy_Pct', 'Women_MSME_Pct', 'Trading_Pct', 'Credit_Invisibility_Score'
]
corr_labels = [
    'Formal\nCredit %', 'GST\nCompliance', 'E-comm\nInteg',
    'UPI\nAdoption', 'Internet\nPenetr', 'Bank\nBranches',
    'CD\nRatio', 'Literacy\n%', 'Urban\n%', 'GSDP\nIndex',
    'NPA\n%', 'Women\nMSME%', 'Trading\n%', 'Invis.\nScore'
]

corr = df[corr_cols].corr()
cmap_div = LinearSegmentedColormap.from_list(
    'diverging', [C['coral'], C['bg'], C['teal']], N=256
)

fig, ax = plt.subplots(figsize=(14, 12), facecolor=C['bg'])
ax.set_facecolor(C['bg'])

im = ax.imshow(corr.values, cmap=cmap_div, vmin=-1, vmax=1, aspect='auto')

n = len(corr_cols)
for i in range(n):
    for j in range(n):
        v        = corr.values[i, j]
        txt_color = C['white'] if abs(v) > 0.45 else C['muted']
        fw        = 'bold'   if abs(v) > 0.65 else 'normal'
        ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                fontsize=7.5, color=txt_color, fontweight=fw)

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(corr_labels, fontsize=8.5, rotation=0)
ax.set_yticklabels(corr_labels, fontsize=8.5)

cbar = plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.ax.tick_params(colors=C['muted'], labelsize=8)
cbar.set_label('Pearson Correlation', color=C['muted'], fontsize=9)

ax.set_title('Correlation Matrix: Alt Credit Signals × Formal Credit Access\n'
             'Top row = correlation with target variable (Formal Credit Access %)',
             fontsize=13, fontweight='bold', color=C['white'], pad=14)
ax.tick_params(colors=C['muted'])
for sp in ax.spines.values():
    sp.set_edgecolor(C['border'])
add_watermark(ax)
save_chart(fig, 'chart06_correlation_heatmap')

  Rendering Chart 6: Correlation Heatmap...
   ✓ chart06_correlation_heatmap.png  (326 KB)


'/content/outputs/charts/chart06_correlation_heatmap.png'

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CHART 7 — BUBBLE CHART: Invisible MSMEs × CD Ratio × Risk Tier
# Purpose : Show where scale, banking depth, and risk intersect.
# Insight : "Excluded Zone" (low C-D, low access) dominated by Bihar and UP.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('  Rendering Chart 7: Bubble Chart (Invisible MSMEs × CD Ratio)...')

fig, ax = plt.subplots(figsize=(14, 9), facecolor=C['bg'])
ax.set_facecolor(C['surface'])

for tier in ['Critical', 'High', 'Moderate']:
    sub = df[df['Risk_Tier'] == tier]
    ax.scatter(sub['CD_Ratio_2025'], sub['Formal_Credit_Access_Pct'],
               s=sub['Alt_Credit_Invisible_Lakh'] * 5 + 20,
               color=TIER_COLORS[tier], alpha=0.75,
               edgecolors=C['white'], linewidth=0.7, zorder=3,
               label=f'{tier} Risk')

# Annotate states with large invisible MSME counts
for _, row in df.iterrows():
    if row['Alt_Credit_Invisible_Lakh'] > 8:
        ax.annotate(row['State'],
                    (row['CD_Ratio_2025'], row['Formal_Credit_Access_Pct']),
                    xytext=(7, 4), textcoords='offset points',
                    fontsize=8, color=C['text'],
                    arrowprops=dict(arrowstyle='->', color=C['border'], lw=0.7))

# Quadrant dividers
ax.axvline(75, color=C['border'], lw=1, linestyle='--', alpha=0.6)
ax.axhline(22, color=C['border'], lw=1, linestyle='--', alpha=0.6)

quad_style = dict(fontsize=8.5, color=C['muted'], style='italic', alpha=0.8)
ax.text(20,  40, 'LOW C-D + LOW ACCESS\n(Excluded Zone)',       **quad_style)
ax.text(100, 40, 'HIGH C-D + LOW ACCESS\n(Credit Misdirected)', **quad_style)
ax.text(20,  10, 'LOW C-D + OK ACCESS\n(Rare: small states)',   **quad_style)
ax.text(100, 10, 'HIGH C-D + OK ACCESS\n(Target State)',        **quad_style)

# Bubble size legend
for sz, lbl in [(20, '0L'), (45, '10L'), (95, '30L'), (170, '60L')]:
    ax.scatter([], [], s=sz, color=C['muted'], alpha=0.5, label=lbl)

ax.legend(loc='upper right', framealpha=0.2, fontsize=9, ncol=2)
ax.set_xlabel('Credit-Deposit Ratio 2025 (%) — Banking Penetration Proxy',
              fontsize=10, labelpad=8)
ax.set_ylabel('Formal Credit Access % (Actual)', fontsize=10, labelpad=8)
ax.set_title('MSME Credit Invisibility: Scale × Banking Depth × Risk Tier\n'
             'Bubble size = Credit-invisible MSME count (lakh) | Source: RBI Table 154 + SIDBI 2025',
             fontsize=13, fontweight='bold', color=C['white'], pad=14)
ax.grid(alpha=0.2)
styled_spine(ax)
add_watermark(ax)
save_chart(fig, 'chart07_bubble_chart')

  Rendering Chart 7: Bubble Chart (Invisible MSMEs × CD Ratio)...
   ✓ chart07_bubble_chart.png  (223 KB)


'/content/outputs/charts/chart07_bubble_chart.png'

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CHART 8 — K-MEANS CLUSTER ANALYSIS + PCA PROJECTION
# Purpose : Reveal 4 distinct MSME ecosystem archetypes in 2D PCA space.
# Insight : "Systemically Excluded" cluster (amber) has highest credit access
#           paradoxically — these are metro UTs with skewed data.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('  Rendering Chart 8: K-Means Cluster Analysis...')

CLUSTER_COLORS = {
    'Digitally Advanced':     C['lime'],
    'Infrastructure Laggard': C['coral'],
    'Banking Accessible':     C['teal'],
    'Systemically Excluded':  C['amber'],
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor=C['bg'])
fig.subplots_adjust(wspace=0.35)

# Left: PCA 2D scatter
ax = axes[0]
ax.set_facecolor(C['surface'])
for cname, color in CLUSTER_COLORS.items():
    sub = df[df['Cluster_Name'] == cname]
    idx = sub.index
    if len(idx) == 0:
        continue
    ax.scatter(X_pca[idx, 0], X_pca[idx, 1],
               s=70, color=color, alpha=0.85, label=cname,
               edgecolors=C['white'], linewidth=0.5, zorder=3)
    for i in idx:
        if df.loc[i, 'Total_MSMEs_Lakh'] > 20:
            ax.annotate(df.loc[i, 'State'].split()[-1],
                        (X_pca[i, 0], X_pca[i, 1]),
                        xytext=(5, 3), textcoords='offset points',
                        fontsize=7.5, color=color)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=9)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=9)
ax.set_title('PCA Projection of 4 MSME Clusters\n(18-feature space → 2D)',
             fontsize=11, fontweight='bold', color=C['white'], pad=10)
ax.legend(fontsize=8.5, framealpha=0.2)
ax.grid(alpha=0.2)
styled_spine(ax)

# Right: Grouped bar — cluster profiles on 5 signals
ax2 = axes[1]
ax2.set_facecolor(C['surface'])

radar_feats  = ['Formal_Credit_Access_Pct', 'GST_Compliance_Pct',
                'Ecomm_Integration_Pct', 'Internet_Penetration_Pct',
                'Bank_Branches_Per_Lakh']
radar_labels = ['Credit\nAccess', 'GST\nCompl', 'E-comm\nInteg', 'Internet\n%', 'Bank\nBranches']

cluster_means = df.groupby('Cluster_Name')[radar_feats].mean()
x_pos  = np.arange(len(radar_feats))
width  = 0.2

for i, (cname, color) in enumerate(CLUSTER_COLORS.items()):
    if cname not in cluster_means.index:
        continue
    vals = cluster_means.loc[cname].values
    norm = vals / (df[radar_feats].max().values + 1e-9) * 100
    ax2.bar(x_pos + i * width, norm, width, color=color,
            alpha=0.8, edgecolor='none', label=cname)

ax2.set_xticks(x_pos + width * 1.5)
ax2.set_xticklabels(radar_labels, fontsize=9)
ax2.set_ylabel('Normalized Score (0–100)', fontsize=9)
ax2.set_title('Cluster Profile — 5 Key Signals\n(normalized to 0–100 for comparability)',
              fontsize=11, fontweight='bold', color=C['white'], pad=10)
ax2.legend(fontsize=8, framealpha=0.2, loc='upper right')
ax2.grid(axis='y', alpha=0.2)
styled_spine(ax2)

fig.suptitle('K-Means Cluster Analysis: 4 Types of MSME Credit Ecosystems in India',
             fontsize=14, fontweight='bold', color=C['white'], y=1.01)
add_watermark(axes[1])
save_chart(fig, 'chart08_cluster_analysis')

  Rendering Chart 8: K-Means Cluster Analysis...
   ✓ chart08_cluster_analysis.png  (203 KB)


'/content/outputs/charts/chart08_cluster_analysis.png'

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CHART 9 — REGIONAL DEEP DIVE: Boxplot + Strip Overlay (3 metrics)
# Purpose : Compare Credit Invisibility, Credit Access, and E-commerce across regions.
# Insight : South leads in all three; Northeast is worst in all three.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('  Rendering Chart 9: Regional Deep Dive...')

fig, axes = plt.subplots(1, 3, figsize=(18, 7), facecolor=C['bg'])
fig.subplots_adjust(wspace=0.35)

metrics = [
    ('Credit_Invisibility_Score', 'Credit Invisibility Score', 0, 100),
    ('Formal_Credit_Access_Pct',  'Formal Credit Access %',    0,  50),
    ('Ecomm_Integration_Pct',     'E-commerce Integration %',  0,  50),
]
region_order2 = ['South', 'West', 'North', 'Central', 'East', 'Northeast']
r_colors = [REGION_COLORS[r] for r in region_order2]

for ax, (col, label, ymin, ymax) in zip(axes, metrics):
    ax.set_facecolor(C['surface'])
    data_by_region = [df[df['Region'] == r][col].values for r in region_order2]

    bp = ax.boxplot(data_by_region, labels=region_order2, patch_artist=True,
                    medianprops=dict(color=C['white'], linewidth=2.5),
                    whiskerprops=dict(color=C['muted'], linewidth=1),
                    capprops=dict(color=C['muted'], linewidth=1),
                    flierprops=dict(marker='o', color=C['muted'], markersize=4, alpha=0.5))

    for patch, color in zip(bp['boxes'], r_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.55)
        patch.set_edgecolor(color)

    # Strip plot overlay — individual state observations
    for i, (region, color) in enumerate(zip(region_order2, r_colors), 1):
        sub    = df[df['Region'] == region][col].values
        jitter = np.random.uniform(-0.12, 0.12, len(sub))
        ax.scatter([i] * len(sub) + jitter, sub, color=color, s=40,
                   zorder=4, edgecolors=C['white'], linewidth=0.4, alpha=0.9)

    ax.set_ylabel(label, fontsize=9, labelpad=8)
    ax.set_xticklabels(region_order2, fontsize=8.5, rotation=20)
    ax.set_ylim(ymin, ymax + (ymax - ymin) * 0.1)
    ax.set_title(label, fontsize=10, fontweight='bold', color=C['white'], pad=8)
    ax.grid(axis='y', alpha=0.2)
    styled_spine(ax)

fig.suptitle('Regional Distribution of Key Credit Metrics\n'
             'Boxplot + individual state observations overlaid',
             fontsize=14, fontweight='bold', color=C['white'], y=1.02)
add_watermark(axes[2])
save_chart(fig, 'chart09_regional_deepdive')

  Rendering Chart 9: Regional Deep Dive...
   ✓ chart09_regional_deepdive.png  (176 KB)


'/content/outputs/charts/chart09_regional_deepdive.png'

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CHART 10 — E-COMMERCE & GST → CREDIT ACCESS (OLS Regression Proof)
# Purpose : Demonstrate the two key alternative signal relationships with OLS fit.
# Insight : E-comm r=0.89; GST r=0.87 — both with p < 0.0001.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('  Rendering Chart 10: E-commerce & GST → Credit Access...')

fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor=C['bg'])
fig.subplots_adjust(wspace=0.3)

# Left: E-commerce integration vs Credit Access
ax = axes[0]
ax.set_facecolor(C['surface'])

r_val, p_val   = stats.pearsonr(df['Ecomm_Integration_Pct'], df['Formal_Credit_Access_Pct'])
slope, intercept, *_ = stats.linregress(df['Ecomm_Integration_Pct'], df['Formal_Credit_Access_Pct'])
x_range = np.linspace(df['Ecomm_Integration_Pct'].min() - 1,
                       df['Ecomm_Integration_Pct'].max() + 1, 100)
y_fit   = slope * x_range + intercept

ax.fill_between(x_range, y_fit - 3, y_fit + 3, color=C['teal'], alpha=0.12, zorder=1)
ax.plot(x_range, y_fit, color=C['teal'], lw=2.2, zorder=4, label='OLS trend')
tier_c = [TIER_COLORS[t] for t in df['Risk_Tier']]
ax.scatter(df['Ecomm_Integration_Pct'], df['Formal_Credit_Access_Pct'],
           c=tier_c, s=80, alpha=0.85, edgecolors=C['white'], linewidth=0.6, zorder=3)

for _, row in df.iterrows():
    if row['Total_MSMEs_Lakh'] > 18 or row['Ecomm_Integration_Pct'] > 35:
        ax.annotate(row['State'].split()[-1],
                    (row['Ecomm_Integration_Pct'], row['Formal_Credit_Access_Pct']),
                    xytext=(5, 3), textcoords='offset points', fontsize=7.5, color=C['text'])

ax.text(0.05, 0.92, f'Pearson r = {r_val:.3f}  (p = {p_val:.4f})\nSlope = {slope:.2f} pp per % e-comm',
        transform=ax.transAxes, fontsize=9, color=C['lime'],
        bbox=dict(boxstyle='round,pad=0.4', facecolor=C['panel'], alpha=0.85))
patches2 = [mpatches.Patch(color=TIER_COLORS[t], label=t) for t in ['Critical', 'High', 'Moderate']]
ax.legend(handles=patches2, fontsize=8.5, framealpha=0.2)
ax.set_xlabel('E-commerce Integration % of State MSMEs', fontsize=10, labelpad=8)
ax.set_ylabel('Formal Credit Access % (Actual)', fontsize=10, labelpad=8)
ax.set_title('E-commerce Integration → Credit Access\n'
             'ICRIER 2025: e-comm MSMEs are 1.5–2.5× more likely to get credit',
             fontsize=11, fontweight='bold', color=C['white'], pad=10)
ax.grid(alpha=0.2)
styled_spine(ax)

# Right: GST Compliance vs Credit Access
ax2 = axes[1]
ax2.set_facecolor(C['surface'])

r2_val, p2_val   = stats.pearsonr(df['GST_Compliance_Pct'], df['Formal_Credit_Access_Pct'])
slope2, int2, *_ = stats.linregress(df['GST_Compliance_Pct'], df['Formal_Credit_Access_Pct'])
x2 = np.linspace(df['GST_Compliance_Pct'].min() - 0.5, df['GST_Compliance_Pct'].max() + 0.5, 100)
y2 = slope2 * x2 + int2

ax2.fill_between(x2, y2 - 3, y2 + 3, color=C['amber'], alpha=0.12)
ax2.plot(x2, y2, color=C['amber'], lw=2.2, zorder=4)
region_c = [REGION_COLORS[r] for r in df['Region']]
ax2.scatter(df['GST_Compliance_Pct'], df['Formal_Credit_Access_Pct'],
            c=region_c, s=80, alpha=0.85, edgecolors=C['white'], linewidth=0.6, zorder=3)

for _, row in df.iterrows():
    if row['Total_MSMEs_Lakh'] > 18:
        ax2.annotate(row['State'].split()[-1],
                     (row['GST_Compliance_Pct'], row['Formal_Credit_Access_Pct']),
                     xytext=(4, 3), textcoords='offset points', fontsize=7.5, color=C['text'])

ax2.text(0.05, 0.92, f'Pearson r = {r2_val:.3f}  (p = {p2_val:.4f})\nSlope = {slope2:.2f} pp per % GST',
         transform=ax2.transAxes, fontsize=9, color=C['amber'],
         bbox=dict(boxstyle='round,pad=0.4', facecolor=C['panel'], alpha=0.85))
patches3 = [mpatches.Patch(color=REGION_COLORS[r], label=r) for r in REGION_COLORS]
ax2.legend(handles=patches3, fontsize=8, framealpha=0.2, ncol=2)
ax2.set_xlabel('GST Filing Compliance Rate %', fontsize=10, labelpad=8)
ax2.set_ylabel('Formal Credit Access % (Actual)', fontsize=10, labelpad=8)
ax2.set_title('GST Compliance → Credit Access\nFormalized GST behaviour predicts creditworthiness',
              fontsize=11, fontweight='bold', color=C['white'], pad=10)
ax2.grid(alpha=0.2)
styled_spine(ax2)

fig.suptitle('Two Strongest Alternative Credit Signals: Digital Footprint Proof',
             fontsize=14, fontweight='bold', color=C['white'], y=1.01)
add_watermark(axes[1])
save_chart(fig, 'chart10_alt_signals_proof')

  Rendering Chart 10: E-commerce & GST → Credit Access...
   ✓ chart10_alt_signals_proof.png  (339 KB)


'/content/outputs/charts/chart10_alt_signals_proof.png'

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CHART 11 — INVISIBLE MSME COUNT: Waterfall by Region + Top 10 States
# Purpose : Show geographic concentration of credit-invisible MSMEs.
# Insight : Central India (Bihar, UP, MP) accounts for ~40% of the national total.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('  Rendering Chart 11: Invisible MSME Waterfall...')

region_inv  = df.groupby('Region')['Alt_Credit_Invisible_Lakh'].sum().sort_values(ascending=False)
state_top10 = df.nlargest(10, 'Alt_Credit_Invisible_Lakh')[['State', 'Alt_Credit_Invisible_Lakh', 'Region']]

fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor=C['bg'])
fig.subplots_adjust(wspace=0.35)

# Left: Region totals (bar chart)
ax = axes[0]
ax.set_facecolor(C['surface'])

regions  = region_inv.index.tolist()
values   = region_inv.values
colors_w = [REGION_COLORS[r] for r in regions]

for i, (region, val, color) in enumerate(zip(regions, values, colors_w)):
    ax.bar(i, val, color=color, alpha=0.85, edgecolor='none', zorder=3)
    ax.text(i, val + 1.5, f'{val:.0f}L', ha='center', fontsize=10,
            fontweight='bold', color=color)
    ax.text(i, -8, region, ha='center', fontsize=9, color=C['muted'])

ax.axhline(0, color=C['border'], lw=0.8)
ax.set_xlim(-0.6, len(regions) - 0.4)
ax.set_ylim(-15, max(values) * 1.2)
ax.set_xticks([])
ax.set_ylabel('Credit-Invisible MSMEs (Lakh)', fontsize=10, labelpad=8)
ax.set_title(f'Total Credit-Invisible MSMEs by Region\nNational Total: {sum(values):.0f} Lakh',
             fontsize=11, fontweight='bold', color=C['white'], pad=10)
ax.grid(axis='y', alpha=0.2)
styled_spine(ax)

# Right: Top 10 states horizontal bar
ax2 = axes[1]
ax2.set_facecolor(C['surface'])

state_top10_sorted = state_top10.sort_values('Alt_Credit_Invisible_Lakh')
bar_c = [REGION_COLORS[r] for r in state_top10_sorted['Region']]
bars  = ax2.barh(state_top10_sorted['State'], state_top10_sorted['Alt_Credit_Invisible_Lakh'],
                 color=bar_c, height=0.65, edgecolor='none', alpha=0.88, zorder=3)

for bar, row in zip(bars, state_top10_sorted.itertuples()):
    ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
             f'{bar.get_width():.1f}L', va='center', fontsize=9.5,
             fontweight='bold', color=REGION_COLORS[row.Region])

ax2.set_xlabel('Credit-Invisible MSMEs (Lakh)', fontsize=10, labelpad=8)
ax2.set_title('Top 10 States — Absolute Credit Exclusion\nLakh = 100,000 MSME units',
              fontsize=11, fontweight='bold', color=C['white'], pad=10)
ax2.grid(axis='x', alpha=0.2)
styled_spine(ax2)

patches_r = [mpatches.Patch(color=REGION_COLORS[r], label=r) for r in REGION_COLORS]
ax2.legend(handles=patches_r, fontsize=8, framealpha=0.2, loc='lower right')

fig.suptitle("Where Are India's 370 Lakh Credit-Invisible MSMEs?",
             fontsize=14, fontweight='bold', color=C['white'], y=1.01)
add_watermark(axes[1])
save_chart(fig, 'chart11_invisible_msme_waterfall')

  Rendering Chart 11: Invisible MSME Waterfall...
   ✓ chart11_invisible_msme_waterfall.png  (164 KB)


'/content/outputs/charts/chart11_invisible_msme_waterfall.png'

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CHART 12 — SIGNAL STRENGTH HEATMAP (Top 15 Vulnerable States × 6 Signals)
# Purpose : Diagnose exactly which signals are weak for each high-risk state.
# Insight : Most vulnerable states fail on GST compliance AND e-commerce together.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('  Rendering Chart 12: Signal Strength Heatmap...')

top15 = df.nlargest(15, 'Credit_Invisibility_Score').copy()
signals = [
    ('GST_Compliance_Pct',       'GST\nCompliance %'),
    ('Ecomm_Integration_Pct',    'E-comm\nInteg %'),
    ('Internet_Penetration_Pct', 'Internet\nPenetr %'),
    ('Bank_Branches_Per_Lakh',   'Bank\nBranches/L'),
    ('UPI_Adoption_Index',       'UPI\nAdoption'),
    ('GSDP_Per_Capita_Index',    'GSDP\nIndex'),
]

# Normalize each column 0-100 relative to the full 36-state range
heat_data = top15[[s[0] for s in signals]].copy()
for col in heat_data.columns:
    heat_data[col] = (heat_data[col] - df[col].min()) / (df[col].max() - df[col].min()) * 100

fig, ax = plt.subplots(figsize=(13, 9), facecolor=C['bg'])
ax.set_facecolor(C['bg'])

cmap_heat = LinearSegmentedColormap.from_list('heat', [C['critical'], C['amber'], C['lime']], N=256)
im = ax.imshow(heat_data.values, cmap=cmap_heat, aspect='auto', vmin=0, vmax=100)

n_rows, n_cols = heat_data.shape
for i in range(n_rows):
    for j in range(n_cols):
        v       = heat_data.values[i, j]
        txt_col = C['bg'] if v > 60 else C['white']
        ax.text(j, i, f'{v:.0f}', ha='center', va='center',
                fontsize=9, color=txt_col, fontweight='bold')

ax.set_xticks(range(n_cols))
ax.set_xticklabels([s[1] for s in signals], fontsize=10, color=C['muted'])
ax.set_yticks(range(n_rows))

state_labels = [f"{row['State']}  [{row['Credit_Invisibility_Score']:.1f}]"
                for _, row in top15.iterrows()]
ax.set_yticklabels(state_labels, fontsize=9.5, color=C['text'])

cbar = plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.ax.tick_params(colors=C['muted'], labelsize=8)
cbar.set_label('Normalized Signal Strength  (0 = Weakest → 100 = Strongest)',
               color=C['muted'], fontsize=8.5)

ax.set_title('Alternative Credit Signal Strength — 15 Most Vulnerable States\n'
             'Red = weak signal (high credit exclusion risk) | Green = strong signal',
             fontsize=13, fontweight='bold', color=C['white'], pad=14)
for sp in ax.spines.values():
    sp.set_edgecolor(C['border'])
add_watermark(ax)
save_chart(fig, 'chart12_signal_heatmap')

print(f'\n✅ All 12 charts saved to {OUT_C}/')

  Rendering Chart 12: Signal Strength Heatmap...
   ✓ chart12_signal_heatmap.png  (217 KB)

✅ All 12 charts saved to /content/outputs/charts/


---
##  Section 4 — KPI Summary Dashboard (PNG)

Generates a single-page **executive summary PNG** (`outputs/reports/KPI_Summary_Dashboard.png`) suitable for embedding in presentations or the project README.

The dashboard contains:
- **Title & subtitle** row with model metadata
- **10 KPI boxes** (2 rows × 5): key national metrics + model outputs
- **3 mini charts**: Risk tier distribution · Top 5 SHAP signals · Avg score by region
- **Footer** with author attribution

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — KPI SUMMARY DASHBOARD (PNG)
# ─────────────────────────────────────────────────────────────────────────────
section_title('SECTION 4 — KPI SUMMARY DASHBOARD (PNG)')

fig = plt.figure(figsize=(20, 14), facecolor=C['bg'])
gs  = gridspec.GridSpec(4, 5, figure=fig, hspace=0.55, wspace=0.35,
                         left=0.04, right=0.97, top=0.90, bottom=0.04)

# ── Header ────────────────────────────────────────────────────────────────────
ax_title = fig.add_subplot(gs[0, :])
ax_title.set_facecolor(C['bg'])
ax_title.axis('off')
ax_title.text(0.5, 0.75, 'MSME CREDIT INVISIBILITY SCORE — INDIA 2026',
              ha='center', va='center', fontsize=22, fontweight='bold',
              color=C['white'], transform=ax_title.transAxes)
ax_title.text(0.5, 0.25,
              'Alternative Credit Signal Framework | XGBoost + SHAP | CV R² = 0.912 | '
              'Sources: RBI, MSME Ministry, SIDBI, ICRIER 2025',
              ha='center', va='center', fontsize=10, color=C['muted'],
              transform=ax_title.transAxes)
ax_title.axhline(0.08, color=C['lime'], lw=2, xmin=0.1, xmax=0.9, alpha=0.6)

# ── KPI Row 1 ─────────────────────────────────────────────────────────────────
kpis = [
    ('7.94 Cr',  'Total Registered MSMEs',  C['teal'],    'MSME Ministry, Mar 2026'),
    ('99.3%',    'Micro Enterprise Share',   C['amber'],   'MSME Ministry, Mar 2026'),
    ('₹30L Cr',  'MSME Credit Gap',          C['critical'],'SIDBI-Crisil 2025'),
    ('~20%',     'Have Formal Credit',        C['coral'],   'SIDBI / NITI 2025'),
    ('370 Lakh', 'Credit-Invisible MSMEs',   C['purple'],  'Model Output — Apr 2026'),
]

for i, (val, label, color, src) in enumerate(kpis):
    ax = fig.add_subplot(gs[1, i])
    ax.set_facecolor(C['surface'])
    for sp in ax.spines.values():
        sp.set_edgecolor(color)
        sp.set_linewidth(2)
    ax.axis('off')
    ax.text(0.5, 0.68, val,   ha='center', va='center', fontsize=20,
            fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.35, label, ha='center', va='center', fontsize=9.5,
            color=C['text'],  transform=ax.transAxes, wrap=True)
    ax.text(0.5, 0.08, src,   ha='center', va='center', fontsize=6.5,
            color=C['muted'], transform=ax.transAxes, style='italic')

# ── KPI Row 2 ─────────────────────────────────────────────────────────────────
kpis2 = [
    ('67.0',    'RBI FI-Index (Mar 2025)',  C['lime'],   'RBI MCIR Jul 2025'),
    ('80.1%',   'All India CD Ratio 2025',  C['blue'],   'RBI Table 154 (Real)'),
    ('493.22',  'Digital Payments Index',   C['teal'],   'RBI MCIR Jul 2025'),
    ('21/36',   'Critical-Risk States',     C['critical'],'Model: Score ≥ 80'),
    ('0.912',   'Model CV R² (XGBoost)',    C['lime'],   '5-Fold Cross-Validation'),
]

for i, (val, label, color, src) in enumerate(kpis2):
    ax = fig.add_subplot(gs[2, i])
    ax.set_facecolor(C['surface'])
    for sp in ax.spines.values():
        sp.set_edgecolor(color)
        sp.set_linewidth(2)
    ax.axis('off')
    ax.text(0.5, 0.68, val,   ha='center', va='center', fontsize=20,
            fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.35, label, ha='center', va='center', fontsize=9.5,
            color=C['text'],  transform=ax.transAxes)
    ax.text(0.5, 0.08, src,   ha='center', va='center', fontsize=6.5,
            color=C['muted'], transform=ax.transAxes, style='italic')

# ── Bottom Mini Charts ────────────────────────────────────────────────────────

# Mini 1: Risk tier distribution
ax_b1 = fig.add_subplot(gs[3, 0:2])
ax_b1.set_facecolor(C['surface'])
tier_counts = df['Risk_Tier'].value_counts()
tier_order  = ['Critical', 'High', 'Moderate']
tier_c_list = [C['critical'], C['high'], C['moderate']]
vals_tier   = [tier_counts.get(t, 0) for t in tier_order]
bars_t = ax_b1.bar(tier_order, vals_tier, color=tier_c_list, alpha=0.85, edgecolor='none')
for bar, v in zip(bars_t, vals_tier):
    ax_b1.text(bar.get_x() + bar.get_width() / 2, v + 0.3, f'{v} states',
               ha='center', fontsize=10, fontweight='bold', color=bar.get_facecolor())
ax_b1.set_ylabel('Number of States', fontsize=8.5)
ax_b1.set_title('Risk Tier Distribution\n(36 States/UTs)', fontsize=10, color=C['white'], pad=8)
ax_b1.set_ylim(0, max(vals_tier) + 3)
ax_b1.grid(axis='y', alpha=0.2)
styled_spine(ax_b1)

# Mini 2: Top 5 SHAP signals
ax_b2 = fig.add_subplot(gs[3, 2:4])
ax_b2.set_facecolor(C['surface'])
top5_shap = shap_df.head(5)
labels5   = [FEAT_LABELS.get(f, f) for f in top5_shap['Feature']]
vals5     = top5_shap['Mean_SHAP'].values
pct5      = vals5 / vals5.sum() * 100
colors5   = [C['teal'], C['lime'], C['amber'], C['coral'], C['purple']]
bars5 = ax_b2.barh(labels5[::-1], pct5[::-1], color=colors5[::-1],
                   height=0.55, edgecolor='none', alpha=0.88)
for bar, v in zip(bars5, pct5[::-1]):
    ax_b2.text(v + 0.3, bar.get_y() + bar.get_height() / 2,
               f'{v:.1f}%', va='center', fontsize=9, fontweight='bold', color=C['text'])
ax_b2.set_xlabel('% of Total SHAP', fontsize=8.5)
ax_b2.set_title('Top 5 Predictive Signals\n(% of SHAP importance)', fontsize=10, color=C['white'], pad=8)
ax_b2.grid(axis='x', alpha=0.2)
styled_spine(ax_b2)

# Mini 3: Avg invisibility by region
ax_b3 = fig.add_subplot(gs[3, 4])
ax_b3.set_facecolor(C['surface'])
reg_avg = df.groupby('Region')['Credit_Invisibility_Score'].mean().sort_values(ascending=True)
bars_r  = ax_b3.barh(reg_avg.index, reg_avg.values,
                     color=[REGION_COLORS[r] for r in reg_avg.index],
                     height=0.55, edgecolor='none', alpha=0.88)
for bar, v in zip(bars_r, reg_avg.values):
    ax_b3.text(v + 0.2, bar.get_y() + bar.get_height() / 2,
               f'{v:.1f}', va='center', fontsize=8.5, fontweight='bold', color=C['text'])
ax_b3.set_xlabel('Avg Score', fontsize=8.5)
ax_b3.set_title('Avg Invisibility\nby Region', fontsize=10, color=C['white'], pad=8)
ax_b3.grid(axis='x', alpha=0.2)
styled_spine(ax_b3)

# Footer
fig.text(0.5, 0.01,
         'Om Dadhe  |  github.com/OmDadhe  |  om-dadhe-portfolio.vercel.app  |  linkedin.com/in/contactom',
         ha='center', fontsize=8.5, color=C['muted'], style='italic')

kpi_path = os.path.join(OUT_R, 'KPI_Summary_Dashboard.png')
fig.savefig(kpi_path, dpi=180, bbox_inches='tight', facecolor=C['bg'], edgecolor='none')
plt.close(fig)
print('✅ KPI Dashboard saved: KPI_Summary_Dashboard.png')


══════════════════════════════════════════════════════════════════════
  SECTION 4 — KPI SUMMARY DASHBOARD (PNG)
══════════════════════════════════════════════════════════════════════
✅ KPI Dashboard saved: KPI_Summary_Dashboard.png


---
## 📁 Section 5 — Professional Excel Dashboard (5 Sheets)

Builds a fully styled Excel workbook (`outputs/excel/MSME_Credit_Invisibility_Score_Dashboard.xlsx`) with the same dark-mode design system applied via openpyxl.

| Sheet | Contents |
|-------|----------|
| 📊 Overview | Problem statement, KPI blocks (×2 rows), strategy pillars, footer |
| 🎯 Credit Scores | All 36 states × 17 columns, conditional formatting, colour-scaled risk tiers |
| 📈 RBI CD Ratio (Real) | 2004–2025 real RBI data, 5-yr and 20-yr change flags |
| 🔍 SHAP & Models | Feature importance with interpretation + model comparison table |
| 📋 National Data | Ministry Dashboard stats (31-03-2026) with source attribution |

> All styling helpers (`xl_font`, `xl_fill`, `xl_align`, `set_cell`) are defined in this section.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — PROFESSIONAL EXCEL DASHBOARD
# ─────────────────────────────────────────────────────────────────────────────
section_title('SECTION 5 — EXCEL DASHBOARD')
print('  Building professional 5-sheet Excel dashboard...')

wb = Workbook()

# ── Colour constants (hex, no #, for openpyxl) ────────────────────────────────
DARK   = '0D1117'
SURF   = '161B22'
PANEL  = '21262D'
BORD   = '30363D'
LIME_X = '39D353'
TEAL_X = '00BCD4'
AMBR_X = 'F0B429'
CRIT_X = 'EF4444'
HIGH_X = 'F59E0B'
MOD_X  = '10B981'
WHITE_X= 'FFFFFF'
TEXT_X = 'E6EDF3'
MUTED_X= '8B949E'

# ── Style helper functions ────────────────────────────────────────────────────
def xl_font(bold=False, size=10, color=TEXT_X, name='Calibri', italic=False):
    return Font(bold=bold, size=size, color=color, name=name, italic=italic)

def xl_fill(color):
    return PatternFill('solid', fgColor=color)

def xl_align(h='center', v='center', wrap=False):
    return Alignment(horizontal=h, vertical=v, wrap_text=wrap)

def xl_border(color=BORD, style='thin'):
    s = Side(border_style=style, color=color)
    return Border(left=s, right=s, top=s, bottom=s)

def xl_border_bottom(color=LIME_X):
    t = Side(border_style='medium', color=color)
    n = Side(border_style=None)
    return Border(bottom=t, left=n, right=n, top=n)

def set_cell(ws, row, col, value, bold=False, size=10, color=TEXT_X,
             bg=None, halign='center', valign='center', wrap=False,
             italic=False, border=None, num_fmt=None, font_name='Calibri'):
    """Write a value to a worksheet cell with full styling."""
    c = ws.cell(row=row, column=col, value=value)
    c.font      = xl_font(bold=bold, size=size, color=color, name=font_name, italic=italic)
    c.alignment = xl_align(h=halign, v=valign, wrap=wrap)
    if bg:     c.fill   = xl_fill(bg)
    if border: c.border = border
    if num_fmt: c.number_format = num_fmt
    return c

def col_width(ws, col_letter, width):
    ws.column_dimensions[col_letter].width = width

def row_height(ws, row_num, height):
    ws.row_dimensions[row_num].height = height

print('  ✓ Excel style helpers defined')


══════════════════════════════════════════════════════════════════════
  SECTION 5 — EXCEL DASHBOARD
══════════════════════════════════════════════════════════════════════
  Building professional 5-sheet Excel dashboard...
  ✓ Excel style helpers defined


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SHEET 1: COVER / OVERVIEW
# Contains: title block, two KPI rows, problem statement, strategy pillars, footer
# ══════════════════════════════════════════════════════════════════════════════
ws_cover = wb.active
ws_cover.title = '📊 Overview'
ws_cover.sheet_view.showGridLines = False
ws_cover.sheet_view.showRowColHeaders = False

# Full background fill
for row in range(1, 55):
    for col in range(1, 20):
        ws_cover.cell(row=row, column=col).fill = xl_fill(DARK)

# Title block
ws_cover.merge_cells('B2:R2')
set_cell(ws_cover, 2, 2, 'MSME CREDIT INVISIBILITY SCORE',
         bold=True, size=22, color=WHITE_X, bg=DARK, font_name='Calibri')

ws_cover.merge_cells('B3:R3')
set_cell(ws_cover, 3, 2, 'India State-Level Alternative Credit Signal Framework',
         size=13, color=LIME_X, bg=DARK, italic=True)

ws_cover.merge_cells('B4:R4')
set_cell(ws_cover, 4, 2,
         'Author: Om Dadhe  |  Model: XGBoost + SHAP  |  CV R² = 0.912  |  36 States/UTs  |  April 2026',
         size=9, color=MUTED_X, bg=DARK, italic=True)

# Lime divider row
for col in range(2, 19):
    c = ws_cover.cell(row=5, column=col)
    c.fill   = xl_fill(LIME_X)
    c.border = Border(bottom=Side(border_style='medium', color=LIME_X))
row_height(ws_cover, 5, 4)

# KPI Row 1 (rows 7–9)
kpi_configs = [
    ('B', 'E',  '7.94 Crore',  'Total Registered MSMEs',        TEAL_X),
    ('F', 'I',  '₹30L Crore',  'MSME Credit Gap (SIDBI 2025)',   CRIT_X),
    ('J', 'M',  '~20%',        'Have Formal Credit Access',       AMBR_X),
    ('N', 'Q',  '370 Lakh',    'Credit-Invisible MSMEs (Model)',  '7C3AED'),
]
for start_c, end_c, val, label, color in kpi_configs:
    ws_cover.merge_cells(f'{start_c}7:{end_c}7')
    ws_cover.merge_cells(f'{start_c}8:{end_c}8')
    ws_cover.merge_cells(f'{start_c}9:{end_c}9')
    sc = ord(start_c) - ord('A') + 1
    set_cell(ws_cover, 7, sc, val,   bold=True,  size=18, color=color, bg=SURF)
    set_cell(ws_cover, 8, sc, label, size=8.5,   color=TEXT_X, bg=SURF, wrap=True)
    set_cell(ws_cover, 9, sc, '',    bg=color)
    row_height(ws_cover, 9, 3)
for r in [7, 8]: row_height(ws_cover, r, 28)

# KPI Row 2 (rows 11–13)
kpi2_configs = [
    ('B', 'E',  '67.0',    'RBI FI-Index Mar 2025',       LIME_X),
    ('F', 'I',  '80.1%',   'All-India CD Ratio 2025',      TEAL_X),
    ('J', 'M',  '493.22',  'Digital Payments Index',       '2196F3'),
    ('N', 'Q',  '21/36',   'States at Critical Risk',      CRIT_X),
]
for start_c, end_c, val, label, color in kpi2_configs:
    ws_cover.merge_cells(f'{start_c}11:{end_c}11')
    ws_cover.merge_cells(f'{start_c}12:{end_c}12')
    ws_cover.merge_cells(f'{start_c}13:{end_c}13')
    sc = ord(start_c) - ord('A') + 1
    set_cell(ws_cover, 11, sc, val,   bold=True, size=18, color=color, bg=SURF)
    set_cell(ws_cover, 12, sc, label, size=8.5,  color=TEXT_X, bg=SURF, wrap=True)
    set_cell(ws_cover, 13, sc, '',    bg=color)
    row_height(ws_cover, 13, 3)
for r in [11, 12]: row_height(ws_cover, r, 28)

# Problem statement section (rows 16–22)
ws_cover.merge_cells('B16:R16')
set_cell(ws_cover, 16, 2, '  WHY THIS MATTERS — THE CREDIT INVISIBILITY CRISIS',
         bold=True, size=12, color=DARK, bg=AMBR_X, halign='left')
row_height(ws_cover, 16, 22)

problems = [
    ('B', 'R', 18, '→  India has 7.94 crore registered MSMEs — but 80% have NEVER accessed formal credit'),
    ('B', 'R', 19, '→  Traditional banks reject them for lacking CIBIL scores, audited financials, and collateral'),
    ('B', 'R', 20, '→  The credit gap is ₹30 lakh crore — not because MSMEs are risky, but because they are INVISIBLE to lenders'),
    ('B', 'R', 21, "→  This project builds India's first alt-signal credit score using GST, e-comm, UPI, and banking data"),
    ('B', 'R', 22, '→  Digital footprint alone explains 63% of variance in formal credit access (SHAP analysis)'),
]
for start_c, end_c, row_n, text in problems:
    ws_cover.merge_cells(f'{start_c}{row_n}:{end_c}{row_n}')
    set_cell(ws_cover, row_n, ord(start_c) - 64, text,
             size=9.5, color=TEXT_X, bg=SURF, halign='left', valign='center')
    row_height(ws_cover, row_n, 18)

# Strategy section (rows 25–37)
ws_cover.merge_cells('B25:R25')
set_cell(ws_cover, 25, 2, '  OUR STRATEGY — 5 ALTERNATIVE SIGNAL PILLARS',
         bold=True, size=12, color=DARK, bg=TEAL_X, halign='left')
row_height(ws_cover, 25, 22)

pillars = [
    ('DIGITAL BEHAVIOUR (63.4%)', 'GST compliance + E-commerce integration + UPI adoption = digital cash-flow trail', LIME_X),
    ('TAX COMPLIANCE (9.4%)',     'GSTR-1 and GSTR-3B filing behaviour replaces income statements for credit assessment', TEAL_X),
    ('INFRASTRUCTURE (11.5%)',    'Internet penetration + bank branch density = credit onboarding readiness', AMBR_X),
    ('MACRO-RESILIENCE (5.0%)',   'GSDP per capita + urban mix + literacy = business environment quality proxy', '7C3AED'),
    ('CREDIT QUALITY (7.4%)',     'NPA proxy + CD ratio trend = historical credit behaviour of the state ecosystem', CRIT_X),
]
for i, (pillar, desc, color) in enumerate(pillars):
    row_n = 27 + i * 2
    ws_cover.merge_cells(f'B{row_n}:E{row_n}')
    ws_cover.merge_cells(f'F{row_n}:R{row_n}')
    set_cell(ws_cover, row_n, 2, pillar, bold=True, size=9.5, color=color, bg=PANEL, halign='left')
    set_cell(ws_cover, row_n, 6, desc,   size=9,    color=TEXT_X, bg=SURF,  halign='left', wrap=True)
    row_height(ws_cover, row_n, 20)

# Footer
ws_cover.merge_cells('B40:R40')
set_cell(ws_cover, 40, 2,
         'Om Dadhe  |  github.com/OmDadhe  |  om-dadhe-portfolio.vercel.app  |  linkedin.com/in/contactom',
         size=9, color=MUTED_X, bg=DARK, italic=True)

for c_letter in 'BCDEFGHIJKLMNOPQR':
    col_width(ws_cover, c_letter, 4.5)
col_width(ws_cover, 'B', 5)

print('  ✓ Sheet 1: Overview complete')

  ✓ Sheet 1: Overview complete


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SHEET 2: CREDIT SCORES TABLE (all 36 states, conditional formatting)
# ══════════════════════════════════════════════════════════════════════════════
ws2 = wb.create_sheet('🎯 Credit Scores')
ws2.sheet_view.showGridLines = False

ws2.merge_cells('A1:Q1')
set_cell(ws2, 1, 1, 'MSME CREDIT INVISIBILITY SCORE — STATE-LEVEL DASHBOARD',
         bold=True, size=14, color=WHITE_X, bg=DARK)
row_height(ws2, 1, 30)

ws2.merge_cells('A2:Q2')
set_cell(ws2, 2, 1,
         'XGBoost Alternative Credit Signal Model | CV R² = 0.912 | Sources: RBI, MSME Ministry, SIDBI 2025',
         size=8.5, color=MUTED_X, bg=SURF, italic=True)
row_height(ws2, 2, 16)

headers = [
    'State / UT', 'Region', 'Risk Tier',
    'Invisibility\nScore', 'Formal Credit\nAccess %', 'Predicted\nAccess %',
    'Invisible\nMSMEs (L)', 'Total MSMEs\n(Lakh)',
    'GST\nCompliance %', 'E-comm\nInteg %', 'Internet\n%',
    'Bank\nBranches/L', 'UPI\nAdoption', 'CD\nRatio 2025',
    'Women\nMSME %', 'Literacy\n%', 'GSDP\nIndex'
]
col_widths_s2 = [22, 12, 12, 12, 14, 14, 14, 14, 14, 12, 10, 12, 11, 13, 12, 10, 11]

for i, (h, w) in enumerate(zip(headers, col_widths_s2), 1):
    set_cell(ws2, 3, i, h, bold=True, size=8.5, color=WHITE_X, bg=PANEL, border=xl_border(), wrap=True)
    ws2.column_dimensions[get_column_letter(i)].width = w
row_height(ws2, 3, 30)

tier_bg = {'Critical': CRIT_X, 'High': HIGH_X, 'Moderate': MOD_X}

data_cols = [
    'State', 'Region', 'Risk_Tier',
    'Credit_Invisibility_Score', 'Formal_Credit_Access_Pct', 'Ensemble_Pred',
    'Alt_Credit_Invisible_Lakh', 'Total_MSMEs_Lakh',
    'GST_Compliance_Pct', 'Ecomm_Integration_Pct', 'Internet_Penetration_Pct',
    'Bank_Branches_Per_Lakh', 'UPI_Adoption_Index', 'CD_Ratio_2025',
    'Women_MSME_Pct', 'Literacy_Rate', 'GSDP_Per_Capita_Index'
]

sorted_ws2 = df.sort_values('Credit_Invisibility_Score', ascending=False)

for row_idx, (_, data_row) in enumerate(sorted_ws2.iterrows()):
    r = row_idx + 4
    row_height(ws2, r, 18)
    bg_row = SURF if row_idx % 2 == 0 else DARK
    tier   = data_row['Risk_Tier']

    for col_idx, col in enumerate(data_cols, 1):
        val   = data_row[col]
        is_text = col in ('State', 'Region', 'Risk_Tier')
        bg    = bg_row
        fg    = TEXT_X

        if col == 'Risk_Tier':
            bg = tier_bg.get(tier, PANEL)
            fg = DARK

        if col == 'Credit_Invisibility_Score':
            fg = tier_bg.get(tier, TEXT_X)

        c = ws2.cell(row=r, column=col_idx,
                     value=round(val, 1) if isinstance(val, float) else val)
        c.font      = xl_font(bold=(col == 'Credit_Invisibility_Score'), size=9, color=fg)
        c.fill      = xl_fill(bg)
        c.alignment = xl_align(h='left' if is_text else 'center', v='center')
        c.border    = xl_border()

# Freeze header rows
ws2.freeze_panes = 'A4'
print('  ✓ Sheet 2: Credit Scores complete')

  ✓ Sheet 2: Credit Scores complete


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SHEET 3: RBI CD RATIO REAL DATA (2004–2025, with change flags)
# ══════════════════════════════════════════════════════════════════════════════
ws3 = wb.create_sheet('📈 RBI CD Ratio (Real)')
ws3.sheet_view.showGridLines = False

ws3.merge_cells('A1:Z1')
set_cell(ws3, 1, 1, 'RBI CREDIT-DEPOSIT RATIO — STATE-WISE (2004–2025)',
         bold=True, size=14, color=WHITE_X, bg=DARK)
row_height(ws3, 1, 28)

ws3.merge_cells('A2:Z2')
set_cell(ws3, 2, 1, 'Source: RBI Table 154 — Credit by Place of Utilisation (Real uploaded data)',
         size=8.5, color=MUTED_X, bg=SURF, italic=True)
row_height(ws3, 2, 16)

# Year columns dynamically from real data
rbi_years_sorted = sorted([c for c in rbi_raw.columns if isinstance(c, int)])

header_row = ['State', 'Region'] + [str(y) for y in rbi_years_sorted] + ['5Y Change', '20Y Change']
for j, h in enumerate(header_row, 1):
    set_cell(ws3, 3, j, h, bold=True, size=8.5, color=WHITE_X, bg=PANEL, border=xl_border())
    ws3.column_dimensions[get_column_letter(j)].width = 9 if j > 2 else (20 if j == 1 else 12)
row_height(ws3, 3, 20)

for row_idx, (_, rbi_row) in enumerate(rbi_raw.iterrows()):
    r = row_idx + 4
    row_height(ws3, r, 16)
    state_name = rbi_row['State']
    bg_row     = SURF if row_idx % 2 == 0 else DARK
    region_val = df[df['State'] == state_name]['Region'].values
    region_str = region_val[0] if len(region_val) > 0 else '—'

    set_cell(ws3, r, 1, state_name, size=9, color=TEXT_X, bg=bg_row,
             halign='left', border=xl_border())
    set_cell(ws3, r, 2, region_str, size=9, color=TEAL_X, bg=bg_row,
             border=xl_border())

    vals_list = []
    for j, yr in enumerate(rbi_years_sorted, 3):
        v = rbi_row[yr] if yr in rbi_row.index else np.nan
        try:
            v_float = float(v)
            vals_list.append(v_float)
            fg = LIME_X if v_float >= 80 else (AMBR_X if v_float >= 60 else CRIT_X)
            c = ws3.cell(row=r, column=j, value=round(v_float, 1))
        except:
            vals_list.append(None)
            fg = MUTED_X
            c = ws3.cell(row=r, column=j, value='—')
        c.font      = xl_font(size=8.5, color=fg)
        c.fill      = xl_fill(bg_row)
        c.alignment = xl_align(h='center', v='center')
        c.border    = xl_border()

    # 5-year and 20-year change flags
    last_col = len(rbi_years_sorted) + 3
    valid_vals = [v for v in vals_list if v is not None]
    if len(valid_vals) >= 5:
        chg5  = round(valid_vals[-1] - valid_vals[-5], 1)
        chg5_txt = f'+{chg5}' if chg5 > 0 else str(chg5)
        fg5  = LIME_X if chg5 > 0 else CRIT_X
        c5 = ws3.cell(row=r, column=last_col, value=chg5_txt)
        c5.font = xl_font(bold=True, size=8.5, color=fg5)
        c5.fill = xl_fill(bg_row)
        c5.alignment = xl_align(h='center', v='center')
        c5.border = xl_border()
    if len(valid_vals) >= 20:
        chg20 = round(valid_vals[-1] - valid_vals[0], 1)
        chg20_txt = f'+{chg20}' if chg20 > 0 else str(chg20)
        fg20 = LIME_X if chg20 > 0 else CRIT_X
        c20 = ws3.cell(row=r, column=last_col + 1, value=chg20_txt)
        c20.font = xl_font(bold=True, size=8.5, color=fg20)
        c20.fill = xl_fill(bg_row)
        c20.alignment = xl_align(h='center', v='center')
        c20.border = xl_border()

ws3.freeze_panes = 'C4'
print('  ✓ Sheet 3: RBI CD Ratio complete')

  ✓ Sheet 3: RBI CD Ratio complete


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SHEET 4: SHAP FEATURE IMPORTANCE + MODEL COMPARISON
# ══════════════════════════════════════════════════════════════════════════════
ws4 = wb.create_sheet('🔍 SHAP & Models')
ws4.sheet_view.showGridLines = False

ws4.merge_cells('A1:H1')
set_cell(ws4, 1, 1, 'SHAP FEATURE IMPORTANCE + MODEL COMPARISON',
         bold=True, size=14, color=WHITE_X, bg=DARK)
row_height(ws4, 1, 28)

ws4.merge_cells('A2:H2')
set_cell(ws4, 2, 1,
         'XGBoost SHAP TreeExplainer | 5-Fold Cross-Validation | n=36 states',
         size=8.5, color=MUTED_X, bg=SURF, italic=True)
row_height(ws4, 2, 16)

shap_hdrs   = ['Rank', 'Feature', 'Readable Name', 'Mean |SHAP|',
               'Signal Pillar', 'Business Interpretation', 'Action for Lenders', 'Priority']
shap_widths = [7, 30, 28, 12, 20, 45, 40, 12]

for i, (h, w) in enumerate(zip(shap_hdrs, shap_widths), 1):
    set_cell(ws4, 4, i, h, bold=True, size=9, color=WHITE_X, bg=PANEL, border=xl_border())
    ws4.column_dimensions[get_column_letter(i)].width = w
row_height(ws4, 4, 20)

signal_pillar_map = {
    'Digital_Footprint_Score':   ('Digital Behaviour', 'Composite digital trail — strongest single predictor of credit access', 'Use as primary underwriting signal for micro-MSME', '★★★★★'),
    'Ecomm_Integration_Pct':     ('Digital Behaviour', 'E-commerce MSMEs have 1.5–2.5× higher credit access (ICRIER 2025)', 'Target Amazon/Flipkart/Meesho seller portfolios first', '★★★★★'),
    'GST_Compliance_Pct':        ('Tax Compliance',    'Regular GSTR-1/3B filings proxy monthly cash-flow visibility', 'Pull GST history via Account Aggregator for all MSME loans', '★★★★★'),
    'UPI_Adoption_Index':        ('Digital Behaviour', 'UPI transaction regularity predicts repayment capacity', 'Use UPI history as collateral-free repayment proxy', '★★★★☆'),
    'Banking_Access_Score':      ('Infrastructure',    'Bank branch density determines credit onboarding feasibility', 'Prioritize BC network expansion in low-branch districts', '★★★☆☆'),
    'Internet_Penetration_Pct':  ('Infrastructure',    'Internet access is prerequisite for digital credit onboarding', 'Infrastructure prerequisite — policy intervention needed', '★★★☆☆'),
    'Bank_Branches_Per_Lakh':    ('Infrastructure',    'Physical banking presence enables MSME credit relationships', 'BC agent density as proxy where branches are thin', '★★★☆☆'),
    'Formalization_Score':       ('Formalization',     'Udyam + GST registration signals business legitimacy', 'Bundle Udyam registration with credit pre-qualification', '★★★★☆'),
    'Credit_Momentum_Score':     ('Credit Quality',    'CD ratio trend 2004–2025 shows directional credit growth', 'Historical trajectory matters as much as current ratio', '★★★☆☆'),
    'GST_Density':               ('Tax Compliance',    'GST taxpayers per lakh MSME = formalization density', 'High density = ready customer pool for MSME credit products', '★★★★☆'),
    'GSDP_Per_Capita_Index':     ('Macro-Resilience',  'Economic size determines MSME market quality', 'Macro context only — not actionable for individual underwriting', '★★☆☆☆'),
    'NPA_Proxy_Pct':             ('Credit Quality',    'Estimated bad loan rate as ecosystem health proxy', 'Use as risk multiplier on portfolio-level pricing', '★★★☆☆'),
    'Business_Resilience_Score': ('Macro-Resilience',  'Composite: literacy + urban mix + GSDP = business env quality', 'State-level context for portfolio risk calibration', '★★☆☆☆'),
    'CD_Ratio_2025':             ('Credit Quality',    'Current banking penetration in the state ecosystem', 'Below 60% = structural credit vacuum, not MSME risk', '★★★☆☆'),
    'Literacy_Rate':             ('Macro-Resilience',  'Higher literacy → better financial documentation quality', 'Correlates with GST compliance and digital adoption', '★★☆☆☆'),
    'Urban_Pct':                 ('Macro-Resilience',  'Urban MSMEs have 2.1× better credit access than rural', 'Rural-specific products needed for peri-urban segments', '★★☆☆☆'),
    'Women_MSME_Pct':            ('Formalization',     '35% credit gap for women-owned MSMEs (NITI Aayog 2025)', 'Women-specific MSME lending programs as market differentiator', '★★★☆☆'),
    'Trading_Pct':               ('Formalization',     'Trading MSMEs have 33% credit gap vs 27% for services', 'GST-first underwriting works especially well for traders', '★★★☆☆'),
}

tier_col_shap = [TEAL_X, LIME_X, AMBR_X, CRIT_X]

for i, (_, shap_row) in enumerate(shap_df.iterrows()):
    r = i + 5
    row_height(ws4, r, 22)
    bg = SURF if i % 2 == 0 else DARK
    feat = shap_row.get('Feature', '')
    info = signal_pillar_map.get(feat, ('Unknown', '—', '—', '—'))
    pillar, interp, action, priority = info
    fg_rank = LIME_X if i < 3 else (AMBR_X if i < 8 else MUTED_X)

    row_data = [
        (str(i + 1),                           fg_rank, True),
        (feat,                                  TEXT_X,  False),
        (FEAT_LABELS.get(feat, feat),           TEAL_X,  False),
        (round(shap_row.get('Mean_SHAP', 0), 4),LIME_X,  True),
        (pillar,                                AMBR_X,  False),
        (interp,                                TEXT_X,  False),
        (action,                                MUTED_X, False),
        (priority,                              LIME_X,  True),
    ]
    for col_i, (v, fg, bold) in enumerate(row_data, 1):
        c = ws4.cell(row=r, column=col_i, value=v)
        c.font      = xl_font(bold=bold, size=9, color=fg)
        c.fill      = xl_fill(bg)
        c.alignment = xl_align(h='center' if col_i in (1, 4, 8) else 'left',
                                v='center', wrap=True)
        c.border    = xl_border()

# Model comparison table
start_r = 5 + len(shap_df) + 3
ws4.merge_cells(f'A{start_r}:H{start_r}')
set_cell(ws4, start_r, 1, 'MODEL COMPARISON — 5-FOLD CROSS-VALIDATION',
         bold=True, size=11, color=AMBR_X, bg=SURF, halign='left')
row_height(ws4, start_r, 22)

m_hdrs = ['Model', 'CV R² Mean', 'CV R² Std', 'Train R²', 'MAE', 'RMSE', 'Best For', 'Notes']
for i, h in enumerate(m_hdrs, 1):
    set_cell(ws4, start_r + 1, i, h, bold=True, size=9, color=WHITE_X, bg=PANEL, border=xl_border())
row_height(ws4, start_r + 1, 20)

y_true = df[TARGET].values
model_data = [
    ('XGBoost',          xgb_cv.mean(),   xgb_cv.std(),   r2_score(y_true, xgb_pred),   mean_absolute_error(y_true, xgb_pred),   np.sqrt(mean_squared_error(y_true, xgb_pred)),   'Non-linear patterns', '★ Primary model'),
    ('Random Forest',    rf_cv.mean(),    rf_cv.std(),    r2_score(y_true, rf_pred),    mean_absolute_error(y_true, rf_pred),    np.sqrt(mean_squared_error(y_true, rf_pred)),    'Variance reduction',  'Robust to outliers'),
    ('Gradient Boosting',gb_cv.mean(),    gb_cv.std(),    r2_score(y_true, gb_pred),    mean_absolute_error(y_true, gb_pred),    np.sqrt(mean_squared_error(y_true, gb_pred)),    'Sequential boosting', 'Good with noise'),
    ('Ridge (Baseline)', ridge_cv.mean(), ridge_cv.std(), r2_score(y_true, ridge_pred), mean_absolute_error(y_true, ridge_pred), np.sqrt(mean_squared_error(y_true, ridge_pred)), 'Linear baseline',     'Interpretable'),
    ('Ensemble (Wtd)',   None,            None,           r2_score(y_true, ensemble_pred), mean_absolute_error(y_true, ensemble_pred), np.sqrt(mean_squared_error(y_true, ensemble_pred)), 'Final scoring', '45% XGB+35% RF+20% GB'),
]

for idx, (name, cv_m, cv_s, tr2, mae, rmse, bf, note) in enumerate(model_data):
    r = start_r + 2 + idx
    row_height(ws4, r, 18)
    bg_m   = SURF if idx % 2 == 0 else DARK
    is_best = name == 'XGBoost'
    fg_m   = LIME_X if is_best else TEXT_X
    vals_m = [name,
              f'{cv_m:.3f}' if cv_m else '(ensemble)',
              f'{cv_s:.3f}' if cv_s else '—',
              f'{tr2:.3f}', f'{mae:.2f}', f'{rmse:.2f}', bf, note]
    for col_i, val in enumerate(vals_m, 1):
        c = ws4.cell(row=r, column=col_i, value=val)
        c.font      = xl_font(bold=is_best, size=9, color=fg_m if col_i == 1 else TEXT_X)
        c.fill      = xl_fill(bg_m)
        c.alignment = xl_align(h='center', v='center')
        c.border    = xl_border()

print('  ✓ Sheet 4: SHAP & Models complete')

  ✓ Sheet 4: SHAP & Models complete


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SHEET 5: NATIONAL DATA (Ministry Dashboard — real statistics)
# ══════════════════════════════════════════════════════════════════════════════
ws5 = wb.create_sheet('📋 National Data')
ws5.sheet_view.showGridLines = False

ws5.merge_cells('A1:E1')
set_cell(ws5, 1, 1, 'MSME MINISTRY DASHBOARD — NATIONAL STATISTICS (As of 31-03-2026)',
         bold=True, size=13, color=WHITE_X, bg=DARK)
row_height(ws5, 1, 28)

ws5.merge_cells('A2:E2')
set_cell(ws5, 2, 1,
         'Source: Ministry of MSME Performance Smartboard | Real data used directly from uploaded document',
         size=8.5, color=MUTED_X, bg=SURF, italic=True)
row_height(ws5, 2, 16)

nat_hdrs   = ['Category', 'Metric', 'Value', 'Source', 'Relevance to Model']
nat_widths = [18, 42, 22, 36, 36]

for i, (h, w) in enumerate(zip(nat_hdrs, nat_widths), 1):
    set_cell(ws5, 4, i, h, bold=True, size=9, color=WHITE_X, bg=PANEL, border=xl_border())
    ws5.column_dimensions[get_column_letter(i)].width = w
row_height(ws5, 4, 20)

nat_data = [
    ('Registration', 'Total MSME Registrations (Udyam + UAP)',       '7,94,25,711',             'MSME Ministry, 31-03-2026',  'Scale of credit exclusion problem'),
    ('Registration', 'Udyam Registrations',                           '4,72,77,069',             'MSME Ministry, 31-03-2026',  'Formalized MSMEs — potential credit seekers'),
    ('Registration', 'UAP (Informal Micro)',                          '3,21,48,642',             'MSME Ministry, 31-03-2026',  'Most credit-invisible segment'),
    ('Enterprise',   'Micro Enterprises',                             '7,88,97,394 (99.3%)',     'MSME Ministry, 31-03-2026',  'Micro_Pct feature — high = more vulnerable'),
    ('Enterprise',   'Small Enterprises',                             '4,91,260 (0.6%)',          'MSME Ministry, 31-03-2026',  'Better credit access benchmark'),
    ('Enterprise',   'Medium Enterprises',                            '37,057 (0.05%)',           'MSME Ministry, 31-03-2026',  'Near-zero credit invisibility'),
    ('Gender',       'Female-owned MSMEs',                            '3,11,41,573 (39.2%)',     'MSME Ministry, 31-03-2026',  'Women_MSME_Pct feature — 35% gap'),
    ('Gender',       'Male-owned MSMEs',                              '4,79,78,498 (60.4%)',     'MSME Ministry, 31-03-2026',  'Baseline credit access group'),
    ('Activity',     'Manufacturing',                                  '1,65,78,457 (20.9%)',     'MSME Ministry, 31-03-2026',  '20% credit gap sector'),
    ('Activity',     'Service',                                        '2,89,23,192 (36.4%)',     'MSME Ministry, 31-03-2026',  '27% credit gap sector (SIDBI)'),
    ('Activity',     'Trading',                                        '3,39,24,062 (42.7%)',     'MSME Ministry, 31-03-2026',  'Trading_Pct — 33% credit gap (SIDBI)'),
    ('Employment',   'Total Employment Generated',                     '35,06,42,612',            'MSME Ministry, 31-03-2026',  'Context: scale of social impact'),
    ('Credit',       'CGTMSE Guarantees Issued',                      '1,39,39,040',             'MSME Ministry, 28-02-2026',  'Policy intervention reach vs gap'),
    ('Credit',       'CGTMSE Guarantee Value',                        '₹13,16,741 Crore',        'MSME Ministry, 28-02-2026',  '₹30L Cr gap vs ₹13L Cr guaranteed'),
    ('Credit',       'MSME Credit Gap (SIDBI)',                       '₹30 lakh crore (~24%)',   'SIDBI-Crisil 2025',          'Model target context'),
    ('Credit',       'Trading Sector Gap',                             '33%',                     'SIDBI Pulse Jun 2025',       'Trading_Pct feature weight'),
    ('Credit',       'Services Sector Gap',                            '27%',                     'SIDBI Pulse Jun 2025',       'Sector-level model calibration'),
    ('Credit',       'Women MSME Credit Gap',                         '35%',                     'NITI Aayog 2025',            'Women_MSME_Pct calibration'),
    ('FI Index',     'RBI FI-Index (Mar 2025)',                       '67.0',                    'RBI MCIR July 2025',         'National financial inclusion baseline'),
    ('FI Index',     'RBI FI-Index (Mar 2024)',                       '64.2',                    'RBI MCIR July 2025',         'YoY improvement: +4.4%'),
    ('Digital',      'RBI Digital Pay Index (Mar 2025)',              '493.22',                  'RBI MCIR July 2025',         'UPI_Adoption_Index calibration source'),
    ('Digital',      'UPI Transactions (daily)',                       '~640 million',             'RBI / NPCI 2025',            'UPI feature context'),
    ('Banking',      'All India CD Ratio 2025',                       '80.1%',                   'RBI Table 154 (Real)',        'CD_Ratio_2025 national benchmark'),
    ('Banking',      'Formal credit-active MSMEs',                    '~19-20%',                 'SIDBI / NITI Aayog 2025',    'Target variable baseline'),
    ('PMEGP',        'Loans Sanctioned (FY22-26)',                    '5,73,917 (₹58,979 Cr)',   'MSME Ministry, 05-03-2026',  'Policy credit reach'),
    ('E-commerce',   'MSME e-comm integration',                       '~12-18%',                 'ICRIER Annual Survey 2025',   'Ecomm_Integration_Pct baseline'),
    ('E-commerce',   'Credit uplift from e-comm',                     '1.5–2.5× more likely',   'ICRIER Annual Survey 2025',   'SHAP finding validation'),
]

cat_colors_map = {
    'Registration': TEAL_X, 'Enterprise': AMBR_X,   'Gender':    '7C3AED',
    'Activity':     LIME_X, 'Employment': '2196F3',  'Credit':    CRIT_X,
    'FI Index':     TEAL_X, 'Digital':    AMBR_X,   'Banking':   LIME_X,
    'PMEGP':        MUTED_X,'E-commerce': CRIT_X
}

for idx, (cat, metric, val, src, rel) in enumerate(nat_data):
    r      = idx + 5
    row_height(ws5, r, 20)
    bg_n   = SURF if idx % 2 == 0 else DARK
    cat_color = cat_colors_map.get(cat, TEXT_X)
    data_n = [(cat, cat_color, True), (metric, TEXT_X, False),
              (val, LIME_X, True),    (src, MUTED_X, False), (rel, TEAL_X, False)]
    for col_i, (v, fg, bold) in enumerate(data_n, 1):
        c = ws5.cell(row=r, column=col_i, value=v)
        c.font      = xl_font(bold=bold, size=9, color=fg)
        c.fill      = xl_fill(bg_n)
        c.alignment = xl_align(h='left' if col_i > 1 else 'center', v='center', wrap=True)
        c.border    = xl_border()

# Save the workbook
excel_path = os.path.join(OUT_E, 'MSME_Credit_Invisibility_Score_Dashboard.xlsx')
wb.save(excel_path)
print('  ✓ Sheet 5: National Data complete')
print(f'\n✅ Excel Dashboard saved: MSME_Credit_Invisibility_Score_Dashboard.xlsx')

  ✓ Sheet 5: National Data complete

✅ Excel Dashboard saved: MSME_Credit_Invisibility_Score_Dashboard.xlsx


---
## 📝 Section 6 — Final Summary & Key Findings

Prints a complete audit of all generated outputs and a summary of the project's key quantitative findings.

### Business Recommendations

**For NBFCs / Fintech Lenders:**
1. **GST-first underwriting** — GSTR-1 filing regularity is a stronger signal than CIBIL for micro-MSME segments. Already accessible via the Account Aggregator framework.
2. **E-commerce seller portfolios** — Amazon, Flipkart, Meesho transaction data provides 24-month cash-flow visibility at near-zero acquisition cost.
3. **Target Bihar and UP** — 68 lakh credit-invisible MSMEs in two states = India's largest untapped addressable market.
4. **UPI as repayment proxy** — Regularity of UPI payment behaviour predicts loan repayment capacity.

**For Policymakers (RBI / Ministry of MSME):**
1. Mandate Account Aggregator integration for all MSME loans under ₹25 lakh.
2. Bundle e-commerce registration with Udyam registration — every MSME on a digital marketplace generates credit-accessible data.
3. State-specific playbooks — Northeast and Central India need infrastructure-first intervention before digital credit can scale.
4. Enable GSTN-CIBIL data bridge with MSME consent under DPDP Act 2023.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 — FINAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
section_title('SECTION 6 — PROJECT SUMMARY')

charts = sorted([f for f in os.listdir(OUT_C) if f.endswith('.png')])
print(f'\n  📊 Charts generated ({len(charts)}):')
for ch in charts:
    print(f'     {ch}')

excel_files = os.listdir(OUT_E)
print(f'\n  📁 Excel files ({len(excel_files)}):')
for f in excel_files:
    sz = os.path.getsize(os.path.join(OUT_E, f)) // 1024
    print(f'     {f}  ({sz} KB)')

reports = os.listdir(OUT_R)
print(f'\n  📋 Reports ({len(reports)}):')
for f in reports:
    sz = os.path.getsize(os.path.join(OUT_R, f)) // 1024
    print(f'     {f}  ({sz} KB)')

r_ecomm, _ = stats.pearsonr(df['Ecomm_Integration_Pct'], df['Formal_Credit_Access_Pct'])
r_gst,   _ = stats.pearsonr(df['GST_Compliance_Pct'],    df['Formal_Credit_Access_Pct'])

print(f"""
  ─────────────────────────────────────────────────────
  MODEL PERFORMANCE SUMMARY
  ─────────────────────────────────────────────────────
  XGBoost CV R²    : {xgb_cv.mean():.3f} ± {xgb_cv.std():.3f}
  Random Forest R² : {rf_cv.mean():.3f} ± {rf_cv.std():.3f}
  Grad. Boost  R²  : {gb_cv.mean():.3f} ± {gb_cv.std():.3f}
  Ensemble Train R²: {r2_score(y_true, ensemble_pred):.3f}

  KEY FINDINGS
  ─────────────────────────────────────────────────────
  Credit-invisible MSMEs  : {df['Alt_Credit_Invisible_Lakh'].sum():.0f} lakh
  Critical-risk states    : {len(df[df['Risk_Tier']=='Critical'])} of 36
  Top SHAP signal         : Digital Footprint Score (63.4% of model)
  E-commerce → Credit r   : {r_ecomm:.3f}  (p < 0.0001)
  GST Compliance → Credit r: {r_gst:.3f}  (p < 0.0001)
  Bihar + UP invisible    : {df[df['State'].isin(['Bihar','Uttar Pradesh'])]['Alt_Credit_Invisible_Lakh'].sum():.0f} lakh
  ─────────────────────────────────────────────────────

✅ ALL OUTPUTS COMPLETE
""")


══════════════════════════════════════════════════════════════════════
  SECTION 6 — PROJECT SUMMARY
══════════════════════════════════════════════════════════════════════

  📊 Charts generated (12):
     chart01_credit_invisibility_ranking.png
     chart02_shap_importance.png
     chart03_rbi_cd_trend.png
     chart04_model_comparison.png
     chart05_cv_robustness.png
     chart06_correlation_heatmap.png
     chart07_bubble_chart.png
     chart08_cluster_analysis.png
     chart09_regional_deepdive.png
     chart10_alt_signals_proof.png
     chart11_invisible_msme_waterfall.png
     chart12_signal_heatmap.png

  📁 Excel files (1):
     MSME_Credit_Invisibility_Score_Dashboard.xlsx  (25 KB)

  📋 Reports (1):
     KPI_Summary_Dashboard.png  (326 KB)

  ─────────────────────────────────────────────────────
  MODEL PERFORMANCE SUMMARY
  ─────────────────────────────────────────────────────
  XGBoost CV R²    : 0.912 ± 0.059
  Random Forest R² : 0.891 ± 0.051
  Grad. Boost  R²  : 0.893 ± 

---

<div style="background:#0d1117; border-radius:8px; padding:24px 28px; border:1px solid #30363d; font-family:'Segoe UI',sans-serif; margin-top:24px;">

<div style="display:flex; justify-content:space-between; align-items:center;">
  <div>
    <p style="color:#39d353; font-size:0.95em; font-weight:600; margin:0 0 4px;">Om Dadhe</p>
    <p style="color:#8b949e; font-size:0.85em; margin:0;">Data &amp; Business Analyst · GITAM University Hyderabad (BTech CSE, May 2026)</p>
    <p style="color:#8b949e; font-size:0.85em; margin:4px 0 0;">Building <strong style="color:#e6edf3;">Shortlisted</strong> — AI-powered placement prep for Tier 2/3 college students</p>
  </div>
  <div style="text-align:right;">
    <a href="https://om-dadhe-portfolio.vercel.app" style="color:#00bcd4; font-size:0.85em;">🌐 Portfolio</a><br/>
    <a href="https://github.com/OmDadhe" style="color:#00bcd4; font-size:0.85em;">🐙 GitHub</a><br/>
    <a href="https://linkedin.com/in/contactom" style="color:#00bcd4; font-size:0.85em;">💼 LinkedIn</a>
  </div>
</div>

<hr style="border:none; border-top:1px solid #30363d; margin:16px 0;"/>

<p style="color:#8b949e; font-size:0.8em; margin:0; text-align:center;">
  <em>If this helped you think differently about MSME credit, star the repo and share it.<br/>
  The ₹30 lakh crore gap closes one dataset at a time.</em>
</p>

</div>

In [ ]:
import shutil

shutil.make_archive(
    'MSME_Credit_Invisibility_Score',
    'zip',
    '/content'
)

'/content/MSME_Credit_Invisibility_Score.zip'

In [ ]:
from google.colab import files
files.download('MSME_Credit_Invisibility_Score.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>